<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açın](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-08/NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html)

# Week 08: Multimodal artificial intelligence and data fusion

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lab.html) · [PDF notes](Week08_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-08/NB08_multimodal_ai_and_data_fusion.ipynb)

</div>

**This notebook has two parts.** In the first, an artificial intelligence (AI) assistant writes the code from a prompt, a written instruction. In the second, you write the code yourself in Python. Both parts reach the same results with the same data, so choose one of them at each practice stop of the lecture page.

**Together with the lecture page.** You come to this Jupyter notebook from the practice stops of the lecture page. At the end of every stop, a link takes you back to the lecture page, to the place where reading continues.

| | For whom? | What you do |
|---|---|---|
| **[Part 1: Prompt path](#part-1)** | Participants who do not write code | An AI assistant writes the code from ready prompts; you run it and check the result. |
| **[Part 2: Python path](#part-2)** | Participants who want to learn Python | You write and run the code step by step, with exercises. |

**Where it runs.** The same notebook runs in Google Colab, with nothing to install, and in Jupyter on your own computer (Anaconda; see SETUP.md in the course folder). Choose the one you prefer. In Colab, save your own copy first with *File → Save a copy in Drive*; otherwise the changes are lost when the tab is closed. The network for the ECG in Section 6 and the two fusion networks in Section 7 train in one to three minutes each on an ordinary laptop; no graphics card is needed. Run Section 0 first, whichever part you choose. It prepares the course files and the packages. A link of the lecture page opens the notebook at a stop or a section: There, run the first cell, the preparation cell or the catch-up cell. It also runs Section 0 if this has not happened in the session.

**Data and privacy.** Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution and with the approval of an ethics committee, and never in a cloud notebook or in an AI chat.

## 0. Setup

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub. The second cell loads the packages, chooses the language of the messages and checks the optional packages of the week. Nothing has to be downloaded: The registry, the chest pain unit of hospitals A and B and their ECGs are drawn by the course package on this computer in less than a minute. 

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic import cohort, followup, triage

try:
    import torch
    import torch.nn as nn
except ImportError as error:
    raise ImportError("PyTorch is not installed. Install it as SETUP.md describes, then restart the kernel.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 9, 0):
    raise ImportError("The course package vclinic is older than this notebook. Update the course folder (git pull, or download the ZIP file again) and restart the kernel.")
teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
for name in []:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        print(f"{name}: not installed (optional; the notebook uses a built-in fallback)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 8.1": (0.675, 0.001),
    "Exercise 8.2": (0.26, 0.001),
    "Exercise 8.3": (1378, 0),
    "Exercise A": (34, 0),
    "Exercise B": (0.837, 0.002),
    "Exercise C": (0.998, 0.01),
    "Exercise D": (0.021, 0.001),
    "Exercise E": (304, 0),
    "Exercise F": (75105, 0),
    "Exercise G": (20833, 0),
    "Exercise H": (0.903, 0.01),
    "Exercise I": (0.105, 0.005),
    "Exercise J": (0.139, 0.01),
    "Step 1.1": (0.849, 0.002),
    "Step 1.2": (0.83, 0.002),
    "Step 1.3": (0.833, 0.002),
    "Step 1.4": (0.041, 0.03),
    "Step 2.1": (0.903, 0.001),
    "Step 2.2": (0.9, 0.002),
    "Step 2.3": (0.922, 0.005),
    "Step 2.4": (2.78, 0.1),
    "Step 3.1": (0.37, 0.05),
    "Step 3.2": (9, 2),
    "Step 3.3": (0.125, 0.01),
})
print("Expected answers are ready.")

In [ ]:
#@title Tables of the exercise prompts (run this cell; the Python path does not need to read it)
def exercise_table(key):
    """The table "data" of the prompt of an exercise, built from the data of the week."""
    g = globals()
    k = key
    need = lambda *names: [g[n] for n in names] if all(n in g for n in names) else (_ for _ in ()).throw(
        RuntimeError("Run the sections before this exercise first."))
    if k in ("8.1", "8.2", "8.3"):
        u = triage.presentations("A")
        return pd.DataFrame({"presentation": u["patient_id"], "troponin after one hour": (u["troponin_1h"] / u["troponin_url"]).round(2),
                             "echocardiogram done": u["echo_done"], "MACE": u["mace30"]})
    if k == "A":
        registry, = need("registry")
        return pd.DataFrame({"patient": registry["patient_id"], "set": registry["set"].map({"training": "training", "validation": "validation", "test": "test"}),
                             "event": registry["event24"]})
    if k in ("B", "C"):
        probs_hf, rte, yr = need("probs_hf", "rte", "yr")
        return pd.DataFrame({"probability, ECG": probs_hf["ECG"][rte], "probability, echocardiogram": probs_hf["echo"][rte],
                             "probability, laboratory": probs_hf["laboratory"][rte], "event": yr[rte]})
    if k == "D":
        A, te = need("A", "te")
        return pd.DataFrame({"HEART": A["heart"].to_numpy()[te], "MACE": A["mace30"].to_numpy()[te]})
    if k == "H":
        XE, parts_A, early, tr, te, y = need("XE", "parts_A", "early", "tr", "te", "y")
        XE_ = XE.copy()
        cols = parts_A[4].columns
        XE_[cols] = XE.loc[tr, cols].mean().to_numpy()
        return pd.DataFrame({"probability without the note": early.predict_proba(XE_[te])[:, 1], "MACE": y[te]})
    if k == "I":
        pB_new, rest, yB = need("pB_new", "rest", "yB")
        return pd.DataFrame({"probability": pB_new[rest], "MACE": yB[rest]})
    if k == "J":
        P, te, y = need("P", "te", "y")
        return pd.DataFrame({"probability": P["early"][te], "MACE": y[te]})
    raise KeyError(k)
print("The tables of the exercise prompts are ready.")

<a id="part-1"></a>

# Part 1: Prompt path

**For whom?** This part is for participants who do not write code. Here you type no code. An artificial intelligence (AI) assistant writes it from a ready prompt, a written instruction; you run the code and check the result.

**What you need.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab: Open it with the Gemini button at the top of the window. ChatGPT, Claude, Copilot or Gemini in another browser tab work as well.

**How a step works.**

1. Read the goal of the step.
2. Run the data cell under the prompt. It prepares the table "data" that the prompt describes and shows its first and last rows.
3. Copy the prompt: Select it with the mouse and press Ctrl+C (Cmd+C on a Mac).
4. Paste it into the AI assistant and send it.
5. Copy the code that the assistant returns, paste it into the cell under the data cell in place of the line that starts with `#`, and run the cell with the ▶ button or with Shift+Enter.
6. Run the check cell below it. It prints *correct ✓* or *not yet ✗*.
7. If it prints *not yet*, send the repair prompt below to the assistant together with the code. If the result is still wrong, run the *Ready code* cell. It always gives the correct result, so the next step can start.

**"data" and "answer".** As in Week 1, the data cell of each step prepares the table "data" that the prompt describes. The assistant keeps the result under the name "answer", and the check cell reads it from there.

**At the start of every stop** run its preparation cell once. It loads the data of the stop and also runs Section 0 if this has not happened in the session, so a stop can also be started on another day or straight from the lecture page.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-08/PROMPTS.md), where GitHub shows a copy button next to each one.

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

## Stop 1: Heart failure events from the ECG, the echocardiogram and the laboratory

*Lecture: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#stop-1)* · *The same result on the Python path: Part 2, Sections 1 to 3*

This stop is the mini project of the session. The patients of the registry of the virtual clinic are described by three modalities: the ECG, the echocardiogram and the laboratory values. A model is built for each modality alone and for all three together, early and late fusion are compared, and the last step asks how sure the difference between fusion and the best single modality is.

**At the end of this stop:** The test AUC of the laboratory values alone (0.849), of early fusion of the three modalities (0.830) and of late fusion by the mean of the logits (0.833), and the share of bootstrap samples of the test patients in which early fusion does better than the laboratory alone (0.041).

In [ ]:
#@title Preparation of Stop 1: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
if "registry" not in globals():
    registry = followup.add_followup(cohort.make_cohort(3600))
    # an event within 24 months; patients followed for less than 24 months without an event are left out
    registry = registry[(registry["event"] == 1) | (registry["followup_months"] >= 24)].reset_index(drop=True)
    registry["event24"] = ((registry["event"] == 1) & (registry["followup_months"] <= 24)).astype(int)
    number = registry["patient_id"].str[3:].astype(int)
    registry["set"] = np.select([number <= 2160, number <= 2880], ["training", "validation"], "test")
    registry["lbbb"] = (registry["ecg_conduction"] == "LBBB").astype(int)
    registry["af_ecg"] = (registry["ecg_rhythm"] == "AF").astype(int)
    registry["log_ntprobnp"] = np.log(registry["ntprobnp"])
    registry["log_troponin"] = np.log(registry["troponin"])
    HF_GROUPS = {
        "ECG": ["ecg_hr", "ecg_qrs", "ecg_qt", "lbbb", "af_ecg"],
        "echo": ["lvef", "lvedd_mm", "lavi", "e_e_prime", "tapse_mm", "lv_mass_index"],
        "laboratory": ["log_ntprobnp", "egfr", "log_troponin"],
    }
_SETS = {"training": "training", "validation": "validation", "test": "test"}


def _registry_table():
    """The patients of the registry with their set, the values of the three modalities and the event within 24 months."""
    r = registry
    return pd.DataFrame({"patient": r["patient_id"], "set": r["set"].map(_SETS),
                         "heart rate": r["ecg_hr"], "QRS": r["ecg_qrs"], "QT": r["ecg_qt"],
                         "left bundle branch block": r["lbbb"], "atrial fibrillation": r["af_ecg"],
                         "ejection fraction": r["lvef"], "end-diastolic diameter": r["lvedd_mm"],
                         "left atrial volume index": r["lavi"], "E/e'": r["e_e_prime"], "TAPSE": r["tapse_mm"],
                         "LV mass index": r["lv_mass_index"],
                         "NT-proBNP": r["ntprobnp"], "eGFR": r["egfr"], "troponin": r["troponin"], "event": r["event24"]})


print(len(registry), "patients,", int(registry["event24"].sum()), "events within 24 months")
print("Stop 1 is ready.")

### Step 1.1: The laboratory values alone

**Goal.** Build a first model for heart failure events from the three laboratory values.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how well NT-proBNP, eGFR and troponin alone predict death or a hospitalisation for heart failure within 24 months, as a baseline for models that add the ECG and the echocardiogram.  
> Data: The notebook holds a table named "data" with 3,227 patients of the registry of the virtual clinic, a synthetic cohort: the patients who were followed for at least 24 months or had an event before. Each row is a patient: "patient" gives the patient, "set" the set ("training", 1,936 patients; "validation", 640; "test", 651), and "event" is 1 when the patient died or was admitted to hospital for heart failure within 24 months and 0 otherwise. The ECG gives "heart rate" (per minute), "QRS" and "QT" (ms), "left bundle branch block" and "atrial fibrillation" (1 when present). The echocardiogram gives "ejection fraction" (%), "end-diastolic diameter" (mm), "left atrial volume index" (mL/m²), "E/e'", "TAPSE" (mm) and "LV mass index" (g/m²). The laboratory gives "NT-proBNP" (pg/mL), "eGFR" (mL/min/1.73 m²) and "troponin" (high-sensitivity, ng/L).  
> Task: Use the logarithm of NT-proBNP and of troponin, since both are strongly skewed. Use the three laboratory values as the features. Standardise the columns with the training patients and fit a logistic regression on the training patients. Choose the amount of regularisation by the area under the ROC curve (AUC) on the validation patients, trying 0.01, 0.1, 1 and 10 for the inverse of its strength, and compute the AUC of the chosen model on the test patients.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the chosen strength and the validation and test AUC, and keep the test AUC, with three decimals, under the name "answer".  
> Test: The AUC must lie between 0 and 1, and no test patient may be used before the last step.  

In [ ]:
#@title Data of step 1.1: run this cell (it prepares the table "data" and shows it)
data = _registry_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.1
teach.check_step("Step 1.1", "answer", globals())

In [ ]:
#@title Ready code of step 1.1 (run it only if the code of the assistant does not pass the check)
data = _registry_table()
d = data.copy()
d["log NT-proBNP"] = np.log(d["NT-proBNP"])
d["log troponin"] = np.log(d["troponin"])
groups = {"ECG": ["heart rate", "QRS", "QT", "left bundle branch block", "atrial fibrillation"],
                  "echo": ["ejection fraction", "end-diastolic diameter", "left atrial volume index", "E/e'", "TAPSE", "LV mass index"],
                  "laboratory": ["log NT-proBNP", "eGFR", "log troponin"]}
sets = d["set"].to_numpy()
tr_, va_, te_ = sets == "training", sets == "validation", sets == "test"
y = d["event"].to_numpy()


def fit(columns):
    X = d[columns].to_numpy(dtype=float)
    best = None
    for C in (0.01, 0.1, 1.0, 10.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[tr_], y[tr_])
        auc_val = roc_auc_score(y[va_], model.predict_proba(X[va_])[:, 1])
        if best is None or auc_val > best[0]:
            best = (auc_val, C, model)
    return best[2].predict_proba(X)[:, 1]

p_lab = fit(groups["laboratory"])
print("test AUC", round(roc_auc_score(y[te_], p_lab[te_]), 3))
answer = round(float(roc_auc_score(y[te_], p_lab[te_])), 3)
teach.check_step("Step 1.1", "answer", globals())

**What the result shows.** Three laboratory values give a test AUC of 0.849. NT-proBNP, a peptide that the failing ventricle releases under strain, carries most of it.

### Step 1.2: Early fusion of three modalities

**Goal.** Put the 14 values of the three modalities side by side and fit one model on them.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether a model that sees the ECG, the echocardiogram and the laboratory values of a patient together predicts heart failure events better than the laboratory values alone.  
> Data: The notebook holds a table named "data" with 3,227 patients of the registry of the virtual clinic, a synthetic cohort: the patients who were followed for at least 24 months or had an event before. Each row is a patient: "patient" gives the patient, "set" the set ("training", 1,936 patients; "validation", 640; "test", 651), and "event" is 1 when the patient died or was admitted to hospital for heart failure within 24 months and 0 otherwise. The ECG gives "heart rate" (per minute), "QRS" and "QT" (ms), "left bundle branch block" and "atrial fibrillation" (1 when present). The echocardiogram gives "ejection fraction" (%), "end-diastolic diameter" (mm), "left atrial volume index" (mL/m²), "E/e'", "TAPSE" (mm) and "LV mass index" (g/m²). The laboratory gives "NT-proBNP" (pg/mL), "eGFR" (mL/min/1.73 m²) and "troponin" (high-sensitivity, ng/L).  
> Task: Use the logarithm of NT-proBNP and of troponin, since both are strongly skewed. The three modalities are the ECG (heart rate, QRS, QT, left bundle branch block, atrial fibrillation), the echocardiogram (ejection fraction, end-diastolic diameter, left atrial volume index, E/e', TAPSE, LV mass index) and the laboratory (NT-proBNP, eGFR, troponin). Use all 14 values together as the features of one model. Standardise the columns with the training patients and fit a logistic regression on the training patients. Choose the amount of regularisation by the area under the ROC curve (AUC) on the validation patients, trying 0.01, 0.1, 1 and 10 for the inverse of its strength, and compute the AUC of the chosen model on the test patients.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the chosen strength and the validation and test AUC, and keep the test AUC, with three decimals, under the name "answer".  
> Test: The model must use 14 features, and the AUC must lie between 0 and 1.  

In [ ]:
#@title Data of step 1.2: run this cell (it prepares the table "data" and shows it)
data = _registry_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.2
teach.check_step("Step 1.2", "answer", globals())

In [ ]:
#@title Ready code of step 1.2 (run it only if the code of the assistant does not pass the check)
data = _registry_table()
d = data.copy()
d["log NT-proBNP"] = np.log(d["NT-proBNP"])
d["log troponin"] = np.log(d["troponin"])
groups = {"ECG": ["heart rate", "QRS", "QT", "left bundle branch block", "atrial fibrillation"],
                  "echo": ["ejection fraction", "end-diastolic diameter", "left atrial volume index", "E/e'", "TAPSE", "LV mass index"],
                  "laboratory": ["log NT-proBNP", "eGFR", "log troponin"]}
sets = d["set"].to_numpy()
tr_, va_, te_ = sets == "training", sets == "validation", sets == "test"
y = d["event"].to_numpy()


def fit(columns):
    X = d[columns].to_numpy(dtype=float)
    best = None
    for C in (0.01, 0.1, 1.0, 10.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[tr_], y[tr_])
        auc_val = roc_auc_score(y[va_], model.predict_proba(X[va_])[:, 1])
        if best is None or auc_val > best[0]:
            best = (auc_val, C, model)
    return best[2].predict_proba(X)[:, 1]

p_early = fit(sum(groups.values(), []))
print("features", len(sum(groups.values(), [])), "| test AUC", round(roc_auc_score(y[te_], p_early[te_]), 3))
answer = round(float(roc_auc_score(y[te_], p_early[te_])), 3)
teach.check_step("Step 1.2", "answer", globals())

**What the result shows.** Early fusion gives a test AUC of 0.830, against 0.849 for the laboratory alone. Eleven more values have not made the model better here.

### Step 1.3: Late fusion by the mean of the logits

**Goal.** Fit one model for each modality and combine their predictions by the mean of their logits.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compare a single model on all values with three separate models, one for each modality, whose opinions are averaged.  
> Data: The notebook holds a table named "data" with 3,227 patients of the registry of the virtual clinic, a synthetic cohort: the patients who were followed for at least 24 months or had an event before. Each row is a patient: "patient" gives the patient, "set" the set ("training", 1,936 patients; "validation", 640; "test", 651), and "event" is 1 when the patient died or was admitted to hospital for heart failure within 24 months and 0 otherwise. The ECG gives "heart rate" (per minute), "QRS" and "QT" (ms), "left bundle branch block" and "atrial fibrillation" (1 when present). The echocardiogram gives "ejection fraction" (%), "end-diastolic diameter" (mm), "left atrial volume index" (mL/m²), "E/e'", "TAPSE" (mm) and "LV mass index" (g/m²). The laboratory gives "NT-proBNP" (pg/mL), "eGFR" (mL/min/1.73 m²) and "troponin" (high-sensitivity, ng/L).  
> Task: Use the logarithm of NT-proBNP and of troponin, since both are strongly skewed. The three modalities are the ECG (heart rate, QRS, QT, left bundle branch block, atrial fibrillation), the echocardiogram (ejection fraction, end-diastolic diameter, left atrial volume index, E/e', TAPSE, LV mass index) and the laboratory (NT-proBNP, eGFR, troponin). For each modality separately: standardise the columns with the training patients and fit a logistic regression on the training patients. Choose the amount of regularisation by the area under the ROC curve (AUC) on the validation patients, trying 0.01, 0.1, 1 and 10 for the inverse of its strength, and compute the AUC of the chosen model on the test patients. Then turn the three probabilities of every patient into logits, log(p / (1 − p)), take their mean, turn the mean back into a probability and compute its AUC on the test patients.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the test AUC of each modality and of their late fusion, and keep the AUC of the late fusion, with three decimals, under the name "answer".  
> Test: There must be three models, one for each modality, and the AUC must lie between 0 and 1.  

In [ ]:
#@title Data of step 1.3: run this cell (it prepares the table "data" and shows it)
data = _registry_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.3
teach.check_step("Step 1.3", "answer", globals())

In [ ]:
#@title Ready code of step 1.3 (run it only if the code of the assistant does not pass the check)
data = _registry_table()
d = data.copy()
d["log NT-proBNP"] = np.log(d["NT-proBNP"])
d["log troponin"] = np.log(d["troponin"])
groups = {"ECG": ["heart rate", "QRS", "QT", "left bundle branch block", "atrial fibrillation"],
                  "echo": ["ejection fraction", "end-diastolic diameter", "left atrial volume index", "E/e'", "TAPSE", "LV mass index"],
                  "laboratory": ["log NT-proBNP", "eGFR", "log troponin"]}
sets = d["set"].to_numpy()
tr_, va_, te_ = sets == "training", sets == "validation", sets == "test"
y = d["event"].to_numpy()


def fit(columns):
    X = d[columns].to_numpy(dtype=float)
    best = None
    for C in (0.01, 0.1, 1.0, 10.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[tr_], y[tr_])
        auc_val = roc_auc_score(y[va_], model.predict_proba(X[va_])[:, 1])
        if best is None or auc_val > best[0]:
            best = (auc_val, C, model)
    return best[2].predict_proba(X)[:, 1]

logits = []
for name, columns in groups.items():
    p_m = fit(columns)
    print(name, round(roc_auc_score(y[te_], p_m[te_]), 3))
    logits.append(np.log(p_m / (1 - p_m)))
p_late = 1 / (1 + np.exp(-np.mean(logits, axis=0)))
print("late fusion", round(roc_auc_score(y[te_], p_late[te_]), 3))
answer = round(float(roc_auc_score(y[te_], p_late[te_])), 3)
teach.check_step("Step 1.3", "answer", globals())

**What the result shows.** The mean of the three logits gives 0.833, close to early fusion (0.830). When one modality carries most of the information, the way of fusing matters little.

### Step 1.4: How sure is the difference?

**Goal.** Draw the test patients again and again with replacement and see how often early fusion beats the laboratory alone.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether the difference between a model of three modalities and the laboratory values alone, measured on a few hundred test patients with few events, would hold on other patients.  
> Data: The notebook holds a table named "data" with 3,227 patients of the registry of the virtual clinic, a synthetic cohort: the patients who were followed for at least 24 months or had an event before. Each row is a patient: "patient" gives the patient, "set" the set ("training", 1,936 patients; "validation", 640; "test", 651), and "event" is 1 when the patient died or was admitted to hospital for heart failure within 24 months and 0 otherwise. The ECG gives "heart rate" (per minute), "QRS" and "QT" (ms), "left bundle branch block" and "atrial fibrillation" (1 when present). The echocardiogram gives "ejection fraction" (%), "end-diastolic diameter" (mm), "left atrial volume index" (mL/m²), "E/e'", "TAPSE" (mm) and "LV mass index" (g/m²). The laboratory gives "NT-proBNP" (pg/mL), "eGFR" (mL/min/1.73 m²) and "troponin" (high-sensitivity, ng/L).  
> Task: Use the logarithm of NT-proBNP and of troponin, since both are strongly skewed. The three modalities are the ECG (heart rate, QRS, QT, left bundle branch block, atrial fibrillation), the echocardiogram (ejection fraction, end-diastolic diameter, left atrial volume index, E/e', TAPSE, LV mass index) and the laboratory (NT-proBNP, eGFR, troponin). Fit the model of the laboratory values and the model of all 14 values as follows. Standardise the columns with the training patients and fit a logistic regression on the training patients. Choose the amount of regularisation by the area under the ROC curve (AUC) on the validation patients, trying 0.01, 0.1, 1 and 10 for the inverse of its strength, and compute the AUC of the chosen model on the test patients. Then draw 2,000 bootstrap samples, each drawing as many test patients as there are with replacement. Fix the random seed to 2026 so that the result can be repeated. In every sample that contains patients with and without an event, compute the AUC of both models and the difference, early fusion minus the laboratory. Report the mean difference, its 95 percent interval from the 2.5th to the 97.5th percentile, and the share of samples in which the difference is above 0.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the mean difference and its interval, and keep the share, with three decimals, under the name "answer".  
> Test: The share must lie between 0 and 1, and both models must be compared on the same patients of every sample.  

In [ ]:
#@title Data of step 1.4: run this cell (it prepares the table "data" and shows it)
data = _registry_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.4
teach.check_step("Step 1.4", "answer", globals())

In [ ]:
#@title Ready code of step 1.4 (run it only if the code of the assistant does not pass the check)
data = _registry_table()
d = data.copy()
d["log NT-proBNP"] = np.log(d["NT-proBNP"])
d["log troponin"] = np.log(d["troponin"])
groups = {"ECG": ["heart rate", "QRS", "QT", "left bundle branch block", "atrial fibrillation"],
                  "echo": ["ejection fraction", "end-diastolic diameter", "left atrial volume index", "E/e'", "TAPSE", "LV mass index"],
                  "laboratory": ["log NT-proBNP", "eGFR", "log troponin"]}
sets = d["set"].to_numpy()
tr_, va_, te_ = sets == "training", sets == "validation", sets == "test"
y = d["event"].to_numpy()


def fit(columns):
    X = d[columns].to_numpy(dtype=float)
    best = None
    for C in (0.01, 0.1, 1.0, 10.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[tr_], y[tr_])
        auc_val = roc_auc_score(y[va_], model.predict_proba(X[va_])[:, 1])
        if best is None or auc_val > best[0]:
            best = (auc_val, C, model)
    return best[2].predict_proba(X)[:, 1]

p_lab = fit(groups["laboratory"])
p_early = fit(sum(groups.values(), []))
yt, a_, b_ = y[te_], p_early[te_], p_lab[te_]
rng = np.random.default_rng(2026)
diff = []
while len(diff) < 2000:
    i = rng.integers(0, len(yt), len(yt))
    if 0 < yt[i].sum() < len(i):
        diff.append(roc_auc_score(yt[i], a_[i]) - roc_auc_score(yt[i], b_[i]))
diff = np.array(diff)
print("mean difference", round(diff.mean(), 3), "| 95% interval", np.round(np.percentile(diff, [2.5, 97.5]), 3))
answer = round(float((diff > 0).mean()), 3)
teach.check_step("Step 1.4", "answer", globals())

**What the result shows.** Early fusion does better than the laboratory alone in a share of 0.041 of the samples. With a few dozen test events, a difference of this size could go either way; NT-proBNP already summarises much of what the other two modalities show.

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#last-two-hours-cardiovascular-risk-scoring-system) and continue reading at «Last two hours: Cardiovascular risk scoring system». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: Major adverse cardiac events in the chest pain unit: HEART, single modalities and fusion

*Lecture: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#stop-2)* · *The same result on the Python path: Part 2, Sections 4 to 8*

The chest pain unit of the virtual clinic predicts MACE within 30 days. The HEART score sets the bar. A model fuses the clinical table, the troponins and the echocardiogram, a second model stacks five single models, and the last step asks whether the probabilities of two kinds of late fusion can be trusted.

**At the end of this stop:** The test AUC of the HEART score (0.903), of early fusion of the clinical table, the troponins and the echocardiogram (0.900) and of stacking five single models (0.922), and the calibration slope of late fusion by the mean of the logits (2.78).

In [ ]:
#@title Preparation of Stop 2: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
if "chest" not in globals():
    chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
    A = chest[chest["hospital"] == "A"].reset_index(drop=True)
    B = chest[chest["hospital"] == "B"].reset_index(drop=True)
    tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
    y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()
if "fit_logistic" not in globals():
    def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
        """Standardise the columns and fit a logistic regression; with a validation set, keep the inverse regularisation strength with the largest validation AUC."""
        best = None
        for C in (Cs if Xv is not None else (1.0,)):
            model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
            auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
            if best is None or auc > best[1]:
                best = (model, auc)
        return best[0]


    def logit(p):
        """The log-odds of a probability."""
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))


    def bootstrap_auc(y, p, n=2000, seed=2026):
        """AUC in n bootstrap samples of the patients: an array of n values."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p[i]))
        return np.array(out)


    def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
        """The AUC of p1 minus the AUC of p2 in n bootstrap samples of the same patients: an array of n differences."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
        return np.array(out)


    def calibration(y, p):
        """Calibration in the large (the intercept with the slope fixed at 1) and the calibration slope."""
        z = logit(p)
        a = 0.0
        for _ in range(50):
            q = 1 / (1 + np.exp(-(a + z)))
            a += (y - q).sum() / (q * (1 - q)).sum()
        slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
        return a, slope


    def reliability(y, p, bins=10):
        """Mean predicted risk and observed share of events in groups of equal size: two arrays."""
        order = np.argsort(p)
        groups = np.array_split(order, bins)
        return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])
_SETS = {"training": "training", "validation": "validation", "test": "test", "external": "hospital B"}
# sets and outcomes under private names, which the code of an assistant does not overwrite
_tr, _va, _te = [(A["set"] == _s).to_numpy() for _s in ("training", "validation", "test")]
_y, _yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()


def _unit_table(d):
    """The presentations of a hospital with what the team knows at arrival and MACE within 30 days."""
    url = d["troponin_url"]
    return pd.DataFrame({"presentation": d["patient_id"], "set": d["set"].map(_SETS), "age": d["age"],
                         "sex": d["sex"].map({"M": "man", "F": "woman"}),
                         "smoker": d["smoker"], "diabetes": d["diabetes"], "hypertension": d["hypertension"],
                         "hyperlipidaemia": d["hyperlipidaemia"], "obesity": d["obesity"],
                         "family history": d["family_history"], "known coronary artery disease": d["known_cad"],
                         "heart rate": d["heart_rate"], "systolic pressure": d["sbp"], "eGFR": d["egfr"],
                         "troponin at arrival": (d["troponin_0h"] / url).round(2),
                         "troponin after one hour": (d["troponin_1h"] / url).round(2),
                         "echocardiogram done": d["echo_done"], "ejection fraction": d["lvef"],
                         "wall motion abnormality": d["rwma"], "HEART": d["heart"], "MACE": d["mace30"]})


def _course_probabilities():
    """The probabilities of MACE that the models of the course give at hospitals A and B: five single modalities, early fusion and the mean of the logits (computed once, one to three minutes, most of it for the network of the ECG)."""
    if "_prob_cache" in globals():
        return _prob_cache
    tr, va, y = _tr, _va, _y
    CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
                "heart_rate", "sbp", "egfr"]


    def clinical(d):
        """The clinical table: age, sex, risk factors, known coronary artery disease, heart rate, systolic pressure and eGFR."""
        return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


    def laboratory(d):
        """Troponin in multiples of the upper reference limit of its assay: the log of the value at arrival and after one hour, the change in one hour, and whether the second value exists."""
        url = d["troponin_url"]
        t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
        return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                             "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


    def echo(d):
        """The bedside echocardiogram: whether it was done, the ejection fraction (60 when not done) and a regional wall motion abnormality (0 when not done)."""
        return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
    svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


    def note(d):
        """The triage note as 20 numbers: TF-IDF of words and word pairs, reduced by a singular value decomposition fitted on the training notes."""
        return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

    ecg_A = triage.ecg_array(A)          # 4,000 ECGs of 12 leads, 10 seconds at 100 Hz: 4000 x 12 x 1000
    ecg_B = triage.ecg_array(B)


    class ECGNet(nn.Module):
        """A one-dimensional convolutional network for a 12-lead ECG: four blocks, the mean of every channel over time and one output."""
        def __init__(self):
            super().__init__()
            block = lambda a, b: nn.Sequential(nn.Conv1d(a, b, 7, padding=3), nn.BatchNorm1d(b), nn.ReLU(), nn.MaxPool1d(2))
            self.features = nn.Sequential(block(12, 32), block(32, 64), block(64, 64), block(64, 64))
            self.head = nn.Linear(64, 1)

        def embed(self, x):
            return self.features(x).mean(dim=2)

        def forward(self, x):
            return self.head(self.embed(x)).squeeze(1)


    def train_ecg(model, X, y, Xv, yv, epochs=12, seed=2026, verbose=True):
        """Adam with a learning rate of 0.001 in groups of 64 ECGs, each shifted in time by up to half a second; the weights of the epoch with the best validation AUC are kept."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.roll(X[b], int(torch.randint(-50, 51, (1,))), dims=2)
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, ecg_predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    @torch.no_grad()
    def ecg_predict(model, X, batch=256):
        model.eval()
        return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


    @torch.no_grad()
    def ecg_embed(model, X, batch=256):
        model.eval()
        return torch.cat([model.embed(torch.as_tensor(X[i:i + batch])) for i in range(0, len(X), batch)]).numpy()
    torch.manual_seed(RANDOM_STATE)
    net = ECGNet()
    train_ecg(net, ecg_A[tr], y[tr], ecg_A[va], y[va], verbose=False)
    emb = lambda X: pd.DataFrame(ecg_embed(net, X), columns=[f"ecg_{i + 1}" for i in range(64)])
    out = {}
    for name, d, X in (("A", A, ecg_A), ("B", B, ecg_B)):
        parts = {"clinical": clinical(d), "troponin": laboratory(d), "echocardiogram": echo(d), "note": note(d)}
        out[name] = dict(parts=parts, ecg=ecg_predict(net, X), embedding=emb(X))
    p = {"A": {}, "B": {}}
    for k in out["A"]["parts"]:
        m = fit_logistic(out["A"]["parts"][k][tr], y[tr], Xv=out["A"]["parts"][k][va], yv=y[va])
        for h in ("A", "B"):
            p[h][k] = m.predict_proba(out[h]["parts"][k])[:, 1]
    for h in ("A", "B"):
        p[h]["ECG network"] = out[h]["ecg"]
    early = lambda h: pd.concat([out[h]["parts"]["clinical"], out[h]["parts"]["troponin"],
                                 out[h]["parts"]["echocardiogram"], out[h]["embedding"], out[h]["parts"]["note"]], axis=1)
    m = fit_logistic(early("A")[tr], y[tr], Xv=early("A")[va], yv=y[va])
    for h in ("A", "B"):
        p[h]["early"] = m.predict_proba(early(h))[:, 1]
        z = np.column_stack([logit(p[h][k]) for k in ["clinical", "troponin", "echocardiogram", "ECG network", "note"]])
        p[h]["mean"] = 1 / (1 + np.exp(-z.mean(axis=1)))
    globals()["_prob_cache"] = p
    return p


print(len(A), "presentations at hospital A and", len(B), "at hospital B")
print("Stop 2 is ready.")

### Step 2.1: The HEART score

**Goal.** Add up the five items of the HEART score and see how well the total separates the presentations with and without a MACE.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants the result of the HEART score on the test presentations of the chest pain unit, the bar that every model of the unit must clear.  
> Data: The notebook holds a table named "data" with the 800 test presentations of a chest pain unit of the virtual clinic, a synthetic data set. Each row is a presentation: "presentation" gives its number, the five columns "history", "ECG", "age", "risk factors" and "troponin" give the points of the five items of the HEART score (0, 1 or 2 each), and "MACE" is 1 when a major adverse cardiac event (myocardial infarction, urgent revascularisation or death) followed within 30 days and 0 otherwise.  
> Task: Add up the five items into the HEART score (0 to 10). Compute its AUC against the column "MACE". Then form the groups 0 to 3, 4 to 6 and 7 to 10 and give, for each, the number of presentations, its share and the share of MACE in it.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the table of the three groups, and keep the AUC, with three decimals, under the name "answer".  
> Test: Every score must lie between 0 and 10, and the three groups must hold all 800 presentations.  

In [ ]:
#@title Data of step 2.1: run this cell (it prepares the table "data" and shows it)
_t = A[_te].reset_index(drop=True)
data = pd.DataFrame({"presentation": _t["patient_id"], "history": _t["heart_h"], "ECG": _t["heart_e"], "age": _t["heart_a"],
                     "risk factors": _t["heart_r"], "troponin": _t["heart_t"], "MACE": _t["mace30"]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.1
teach.check_step("Step 2.1", "answer", globals())

In [ ]:
#@title Ready code of step 2.1 (run it only if the code of the assistant does not pass the check)
_t = A[_te].reset_index(drop=True)
data = pd.DataFrame({"presentation": _t["patient_id"], "history": _t["heart_h"], "ECG": _t["heart_e"], "age": _t["heart_a"],
                     "risk factors": _t["heart_r"], "troponin": _t["heart_t"], "MACE": _t["mace30"]})
heart = data[["history", "ECG", "age", "risk factors", "troponin"]].sum(axis=1)
group = pd.cut(heart, [-1, 3, 6, 10], labels=["0-3", "4-6", "7-10"])
print(data.assign(group=group).groupby("group", observed=True)["MACE"].agg(["size", "mean"]).round(3))
answer = round(float(roc_auc_score(data["MACE"], heart)), 3)
teach.check_step("Step 2.1", "answer", globals())

**What the result shows.** HEART reaches a test AUC of 0.903 with five items of 0 to 2 points each. Its groups separate the risk well: few events with 0 to 3 points, most with 7 to 10.

### Step 2.2: Early fusion of the clinical table, the troponins and the echocardiogram

**Goal.** Put the clinical values, the troponins and the echocardiogram of every presentation in one row, fill the gaps with a marker, and fit one model.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants a risk model for MACE within 30 days from what the team knows one hour after arrival, without the ECG and the note, keeping the patients whose second troponin or echocardiogram is missing.  
> Data: The notebook holds a table named "data" with the 4,000 presentations with chest pain of a chest pain unit of the virtual clinic, a synthetic data set. Each row is a presentation: "presentation" gives its number and "set" its set ("training", 2,400; "validation", 800; "test", 800). The clinical table gives "age" (years), "sex" ("man" or "woman"), seven risk factors that are 1 when present ("smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family history" and "known coronary artery disease"), "heart rate" (per minute), "systolic pressure" (mmHg) and "eGFR". High-sensitivity troponin T is given in multiples of the upper reference limit of the assay: "troponin at arrival" and "troponin after one hour", which is empty when it was not measured. The bedside echocardiogram gives "echocardiogram done" (1 or 0), and when it was done "ejection fraction" (%) and "wall motion abnormality" (1 when a regional abnormality was seen); both are empty otherwise. "HEART" is the HEART score, and "MACE" is 1 when a major adverse cardiac event followed within 30 days and 0 otherwise.  
> Task: Build the features of every presentation: age, sex as 1 for a man, the seven risk factors, heart rate, systolic pressure and eGFR; the logarithm of the troponin at arrival, the logarithm of the troponin after one hour (the value at arrival when it was not measured), the change in one hour (0 when not measured) and a column that is 1 when the second troponin was measured; whether the echocardiogram was done, the ejection fraction (60 when not done) and the wall motion abnormality (0 when not done). Do not use the HEART score. Standardise the columns with the training presentations and fit a logistic regression on the training presentations. Choose the amount of regularisation by the area under the ROC curve (AUC) on the validation presentations, trying 0.01, 0.1, 1 and 10 for the inverse of its strength, and compute the AUC of the chosen model on the test presentations.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the chosen strength and the validation and test AUC, and keep the test AUC, with three decimals, under the name "answer".  
> Test: All 4,000 presentations must be kept, the model must use 19 features, and the HEART score must not be among them.  

In [ ]:
#@title Data of step 2.2: run this cell (it prepares the table "data" and shows it)
data = _unit_table(A)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.2
teach.check_step("Step 2.2", "answer", globals())

In [ ]:
#@title Ready code of step 2.2 (run it only if the code of the assistant does not pass the check)
data = _unit_table(A)
d = data
sets = d["set"].to_numpy()
tr_, va_, te_ = sets == "training", sets == "validation", sets == "test"
y = d["MACE"].to_numpy()
t0, t1 = d["troponin at arrival"], d["troponin after one hour"]
X = pd.DataFrame({
    "age": d["age"], "man": (d["sex"] == "man").astype(int),
    **{c: d[c] for c in ["smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity",
                                   "family history", "known coronary artery disease", "heart rate", "systolic pressure", "eGFR"]},
    "log troponin at arrival": np.log(t0), "log troponin after one hour": np.log(t1.fillna(t0)),
    "change in one hour": (t1 - t0).fillna(0.0), "second troponin measured": t1.notna().astype(int),
    "echocardiogram done": d["echocardiogram done"],
    "ejection fraction": d["ejection fraction"].fillna(60.0),
    "wall motion abnormality": d["wall motion abnormality"].fillna(0.0)}).to_numpy(dtype=float)
best = None
for C in (0.01, 0.1, 1.0, 10.0):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[tr_], y[tr_])
    auc_val = roc_auc_score(y[va_], model.predict_proba(X[va_])[:, 1])
    if best is None or auc_val > best[0]:
        best = (auc_val, C, model)
p = best[2].predict_proba(X)[:, 1]
print("inverse regularisation strength", best[1], "| validation AUC", round(best[0], 3),
      "| test AUC", round(roc_auc_score(y[te_], p[te_]), 3))
answer = round(float(roc_auc_score(y[te_], p[te_])), 3)
teach.check_step("Step 2.2", "answer", globals())

**What the result shows.** Three modalities fused in one row give a test AUC of 0.900, against 0.903 for HEART. The columns that mark a missing value let the model keep every patient.

### Step 2.3: Stacking five single models

**Goal.** Learn on the validation presentations how much each of five single models should count, and test the combination.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist has five models, each reading one modality of a presentation with chest pain, and wants to combine their opinions into one risk with weights learned from data rather than an equal vote.  
> Data: The notebook holds a table named "data" with the 800 validation and the 800 test presentations of a chest pain unit of the virtual clinic, a synthetic data set. Each row is a presentation: "set" gives its set ("validation" or "test"), and five columns give the probability of a major adverse cardiac event within 30 days that five models of the course give from one modality each, all trained on the training presentations: "probability, clinical", "probability, troponin", "probability, echocardiogram", "probability, ECG network" and "probability, note". "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: Turn the five probabilities of every presentation into logits, log(p / (1 − p)). Fit a logistic regression of MACE on the five logits on the validation presentations only, with the default regularisation. Report its five weights and its intercept, and compute the AUC of its probabilities on the test presentations. Compare it with the mean of the five logits.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the weights, the intercept and the test AUC of stacking and of the mean, and keep the test AUC of stacking, with three decimals, under the name "answer".  
> Test: Stacking must learn on the validation presentations only, and every weight must be printed.  

In [ ]:
#@title Data of step 2.3: run this cell (it prepares the table "data" and shows it)
_p = _course_probabilities()["A"]   # one to three minutes the first time: the models of the course are trained once
_k = _va | _te
data = pd.DataFrame({"set": np.where(_va, "validation", "test")[_k],
                     **{"probability, " + _n: _p[_n][_k] for _n in ["clinical", "troponin", "echocardiogram", "ECG network", "note"]},
                     "MACE": _y[_k]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.3
teach.check_step("Step 2.3", "answer", globals())

In [ ]:
#@title Ready code of step 2.3 (run it only if the code of the assistant does not pass the check)
_p = _course_probabilities()["A"]   # one to three minutes the first time: the models of the course are trained once
_k = _va | _te
data = pd.DataFrame({"set": np.where(_va, "validation", "test")[_k],
                     **{"probability, " + _n: _p[_n][_k] for _n in ["clinical", "troponin", "echocardiogram", "ECG network", "note"]},
                     "MACE": _y[_k]})
cols = [c for c in data.columns if c.startswith("probability")]
p = np.clip(data[cols].to_numpy(), 1e-6, 1 - 1e-6)
z = np.log(p / (1 - p))
y = data["MACE"].to_numpy()
va_, te_ = data["set"].to_numpy() == "validation", data["set"].to_numpy() == "test"
stack = LogisticRegression(max_iter=5000).fit(z[va_], y[va_])
print(dict(zip(cols, np.round(stack.coef_[0], 3))), "| intercept", round(stack.intercept_[0], 3))
auc_mean = roc_auc_score(y[te_], z[te_].mean(axis=1))
auc_stack = roc_auc_score(y[te_], stack.predict_proba(z[te_])[:, 1])
print("mean of the logits", round(auc_mean, 3), "| stacking", round(auc_stack, 3))
answer = round(float(auc_stack), 3)
teach.check_step("Step 2.3", "answer", globals())

**What the result shows.** Stacking reaches a test AUC of 0.922. The troponin gets the largest weight, and the weights add up to more than 1: Stacking adds the evidence of the modalities rather than averaging it.

### Step 2.4: Can the probabilities be trusted?

**Goal.** Judge the calibration of two models with almost the same AUC: early fusion and the mean of the logits.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to tell patients their risk of MACE and to send home those below a threshold, and wants to know whether the probabilities of two models that rank the patients equally well mean what they say.  
> Data: The notebook holds a table named "data" with the 800 test presentations of a chest pain unit of the virtual clinic, a synthetic data set. Each row is a presentation: "probability, early fusion" is the probability of a major adverse cardiac event within 30 days from a model of the course that uses all five modalities together, "probability, mean of the logits" the probability from the mean of the logits of five single-modality models, and "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: For both probability columns, compute the AUC and the mean predicted risk against the share of MACE. Compute the calibration intercept (the shift of the logits that makes the mean predicted risk equal to the observed share, with the slope fixed at 1), the calibration slope (a logistic regression of MACE on the logit of the probability, without regularisation) and the Brier score. Then draw a reliability curve for each: Sort the presentations by the probability, cut them into ten groups of equal size, and plot the mean probability of each group against its share of MACE, with the diagonal.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the AUC, the mean predicted risk, the intercept, the slope and the Brier score of both models, show the two curves, and keep the calibration slope of the mean of the logits, with two decimals, under the name "answer".  
> Test: The two curves must have ten points each, and the Brier scores must lie between 0 and 1.  

In [ ]:
#@title Data of step 2.4: run this cell (it prepares the table "data" and shows it)
_p = _course_probabilities()["A"]   # one to three minutes the first time
data = pd.DataFrame({"probability, early fusion": _p["early"][_te], "probability, mean of the logits": _p["mean"][_te],
                     "MACE": _y[_te]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.4
teach.check_step("Step 2.4", "answer", globals())

In [ ]:
#@title Ready code of step 2.4 (run it only if the code of the assistant does not pass the check)
_p = _course_probabilities()["A"]   # one to three minutes the first time
data = pd.DataFrame({"probability, early fusion": _p["early"][_te], "probability, mean of the logits": _p["mean"][_te],
                     "MACE": _y[_te]})


def calibration(yy, p):
    z = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    a = 0.0
    for _ in range(50):
        q = 1 / (1 + np.exp(-(a + z)))
        a += (yy - q).sum() / (q * (1 - q)).sum()
    slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], yy).coef_[0, 0]
    return a, slope, float(np.mean((p - yy) ** 2))

y = data["MACE"].to_numpy()
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for col in ["probability, early fusion", "probability, mean of the logits"]:
    p = data[col].to_numpy()
    a, s, b = calibration(y, p)
    print(col, "| AUC", round(roc_auc_score(y, p), 3), "| mean", round(p.mean(), 3), "| observed", round(y.mean(), 3),
          "| intercept", round(a, 2), "| slope", round(s, 2), "| Brier", round(b, 3))
    groups = np.array_split(np.argsort(p), 10)
    ax.plot([p[g].mean() for g in groups], [y[g].mean() for g in groups], "o-", label=col)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("mean predicted risk")
ax.set_ylabel("observed share of MACE")
ax.legend(fontsize=7)
plt.show()
answer = round(float(calibration(y, data["probability, mean of the logits"].to_numpy())[1]), 2)
teach.check_step("Step 2.4", "answer", globals())

**What the result shows.** The mean of the logits has a calibration slope of 2.78: Its probabilities crowd toward the middle, so low risks are overestimated and high risks underestimated, although it ranks the patients as well as early fusion.

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#another-hospital-drift-and-recalibration) and continue reading at «Another hospital: Drift and recalibration». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: Triage: another hospital, a threshold for discharge and net benefit

*Lecture: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#stop-3)* · *The same result on the Python path: Part 2, Sections 9 to 10*

A model that leaves its hospital must be checked where it is used. The first step updates the model at hospital B with its first patients. The second chooses a threshold for discharge on the validation presentations and compares it with HEART 0 to 3 on the test presentations. The last asks for the net benefit of the model and of HEART for the decision to admit.

**At the end of this stop:** The shift of the logits that hospital B learns from its first 300 patients (0.37), the number of MACE among the test patients sent home by the threshold of the model (9), and the net benefit of the model at a threshold probability of 20 percent (0.125).

In [ ]:
#@title Preparation of Stop 3: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
if "chest" not in globals():
    chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
    A = chest[chest["hospital"] == "A"].reset_index(drop=True)
    B = chest[chest["hospital"] == "B"].reset_index(drop=True)
    tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
    y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()
if "fit_logistic" not in globals():
    def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
        """Standardise the columns and fit a logistic regression; with a validation set, keep the inverse regularisation strength with the largest validation AUC."""
        best = None
        for C in (Cs if Xv is not None else (1.0,)):
            model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
            auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
            if best is None or auc > best[1]:
                best = (model, auc)
        return best[0]


    def logit(p):
        """The log-odds of a probability."""
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))


    def bootstrap_auc(y, p, n=2000, seed=2026):
        """AUC in n bootstrap samples of the patients: an array of n values."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p[i]))
        return np.array(out)


    def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
        """The AUC of p1 minus the AUC of p2 in n bootstrap samples of the same patients: an array of n differences."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
        return np.array(out)


    def calibration(y, p):
        """Calibration in the large (the intercept with the slope fixed at 1) and the calibration slope."""
        z = logit(p)
        a = 0.0
        for _ in range(50):
            q = 1 / (1 + np.exp(-(a + z)))
            a += (y - q).sum() / (q * (1 - q)).sum()
        slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
        return a, slope


    def reliability(y, p, bins=10):
        """Mean predicted risk and observed share of events in groups of equal size: two arrays."""
        order = np.argsort(p)
        groups = np.array_split(order, bins)
        return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])
_SETS = {"training": "training", "validation": "validation", "test": "test", "external": "hospital B"}
# sets and outcomes under private names, which the code of an assistant does not overwrite
_tr, _va, _te = [(A["set"] == _s).to_numpy() for _s in ("training", "validation", "test")]
_y, _yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()


def _unit_table(d):
    """The presentations of a hospital with what the team knows at arrival and MACE within 30 days."""
    url = d["troponin_url"]
    return pd.DataFrame({"presentation": d["patient_id"], "set": d["set"].map(_SETS), "age": d["age"],
                         "sex": d["sex"].map({"M": "man", "F": "woman"}),
                         "smoker": d["smoker"], "diabetes": d["diabetes"], "hypertension": d["hypertension"],
                         "hyperlipidaemia": d["hyperlipidaemia"], "obesity": d["obesity"],
                         "family history": d["family_history"], "known coronary artery disease": d["known_cad"],
                         "heart rate": d["heart_rate"], "systolic pressure": d["sbp"], "eGFR": d["egfr"],
                         "troponin at arrival": (d["troponin_0h"] / url).round(2),
                         "troponin after one hour": (d["troponin_1h"] / url).round(2),
                         "echocardiogram done": d["echo_done"], "ejection fraction": d["lvef"],
                         "wall motion abnormality": d["rwma"], "HEART": d["heart"], "MACE": d["mace30"]})


def _course_probabilities():
    """The probabilities of MACE that the models of the course give at hospitals A and B: five single modalities, early fusion and the mean of the logits (computed once, one to three minutes, most of it for the network of the ECG)."""
    if "_prob_cache" in globals():
        return _prob_cache
    tr, va, y = _tr, _va, _y
    CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
                "heart_rate", "sbp", "egfr"]


    def clinical(d):
        """The clinical table: age, sex, risk factors, known coronary artery disease, heart rate, systolic pressure and eGFR."""
        return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


    def laboratory(d):
        """Troponin in multiples of the upper reference limit of its assay: the log of the value at arrival and after one hour, the change in one hour, and whether the second value exists."""
        url = d["troponin_url"]
        t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
        return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                             "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


    def echo(d):
        """The bedside echocardiogram: whether it was done, the ejection fraction (60 when not done) and a regional wall motion abnormality (0 when not done)."""
        return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
    svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


    def note(d):
        """The triage note as 20 numbers: TF-IDF of words and word pairs, reduced by a singular value decomposition fitted on the training notes."""
        return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

    ecg_A = triage.ecg_array(A)          # 4,000 ECGs of 12 leads, 10 seconds at 100 Hz: 4000 x 12 x 1000
    ecg_B = triage.ecg_array(B)


    class ECGNet(nn.Module):
        """A one-dimensional convolutional network for a 12-lead ECG: four blocks, the mean of every channel over time and one output."""
        def __init__(self):
            super().__init__()
            block = lambda a, b: nn.Sequential(nn.Conv1d(a, b, 7, padding=3), nn.BatchNorm1d(b), nn.ReLU(), nn.MaxPool1d(2))
            self.features = nn.Sequential(block(12, 32), block(32, 64), block(64, 64), block(64, 64))
            self.head = nn.Linear(64, 1)

        def embed(self, x):
            return self.features(x).mean(dim=2)

        def forward(self, x):
            return self.head(self.embed(x)).squeeze(1)


    def train_ecg(model, X, y, Xv, yv, epochs=12, seed=2026, verbose=True):
        """Adam with a learning rate of 0.001 in groups of 64 ECGs, each shifted in time by up to half a second; the weights of the epoch with the best validation AUC are kept."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.roll(X[b], int(torch.randint(-50, 51, (1,))), dims=2)
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, ecg_predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    @torch.no_grad()
    def ecg_predict(model, X, batch=256):
        model.eval()
        return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


    @torch.no_grad()
    def ecg_embed(model, X, batch=256):
        model.eval()
        return torch.cat([model.embed(torch.as_tensor(X[i:i + batch])) for i in range(0, len(X), batch)]).numpy()
    torch.manual_seed(RANDOM_STATE)
    net = ECGNet()
    train_ecg(net, ecg_A[tr], y[tr], ecg_A[va], y[va], verbose=False)
    emb = lambda X: pd.DataFrame(ecg_embed(net, X), columns=[f"ecg_{i + 1}" for i in range(64)])
    out = {}
    for name, d, X in (("A", A, ecg_A), ("B", B, ecg_B)):
        parts = {"clinical": clinical(d), "troponin": laboratory(d), "echocardiogram": echo(d), "note": note(d)}
        out[name] = dict(parts=parts, ecg=ecg_predict(net, X), embedding=emb(X))
    p = {"A": {}, "B": {}}
    for k in out["A"]["parts"]:
        m = fit_logistic(out["A"]["parts"][k][tr], y[tr], Xv=out["A"]["parts"][k][va], yv=y[va])
        for h in ("A", "B"):
            p[h][k] = m.predict_proba(out[h]["parts"][k])[:, 1]
    for h in ("A", "B"):
        p[h]["ECG network"] = out[h]["ecg"]
    early = lambda h: pd.concat([out[h]["parts"]["clinical"], out[h]["parts"]["troponin"],
                                 out[h]["parts"]["echocardiogram"], out[h]["embedding"], out[h]["parts"]["note"]], axis=1)
    m = fit_logistic(early("A")[tr], y[tr], Xv=early("A")[va], yv=y[va])
    for h in ("A", "B"):
        p[h]["early"] = m.predict_proba(early(h))[:, 1]
        z = np.column_stack([logit(p[h][k]) for k in ["clinical", "troponin", "echocardiogram", "ECG network", "note"]])
        p[h]["mean"] = 1 / (1 + np.exp(-z.mean(axis=1)))
    globals()["_prob_cache"] = p
    return p


print(len(A), "presentations at hospital A and", len(B), "at hospital B")
print("Stop 3 is ready.")

### Step 3.1: An update for another hospital

**Goal.** Shift the logits of the model so that its mean risk matches the share of events of the first 300 patients of hospital B, and check the next 700.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist of hospital B wants to use a model from hospital A, whose patients and practice differ, and wants to correct the level of its risks with the first patients of B before relying on it.  
> Data: The notebook holds a table named "data" with the 1,000 presentations with chest pain of a second hospital, B, of the virtual clinic, a synthetic data set, in the order in which they arrived. Each row is a presentation: "order" gives its place in time, "probability" the probability of a major adverse cardiac event within 30 days from a model that learned at the first hospital, A, and "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: Turn the probabilities into logits. On the first 300 presentations, find the shift that, added to every logit, makes the mean of the shifted probabilities equal to the share of MACE of these 300 (for example by Newton steps). Add the shift to the logits of all presentations. For the remaining 700, report the mean predicted risk before and after the update, the observed share of MACE and the AUC before and after.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the shift, the means, the observed share and the two AUCs, and keep the shift, with two decimals, under the name "answer".  
> Test: Only the first 300 presentations may be used to find the shift, and the AUC must not change.  

In [ ]:
#@title Data of step 3.1: run this cell (it prepares the table "data" and shows it)
_p = _course_probabilities()["B"]   # one to three minutes the first time
data = pd.DataFrame({"order": np.arange(1, len(B) + 1), "probability": _p["early"], "MACE": _yB})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.1
teach.check_step("Step 3.1", "answer", globals())

In [ ]:
#@title Ready code of step 3.1 (run it only if the code of the assistant does not pass the check)
_p = _course_probabilities()["B"]   # one to three minutes the first time
data = pd.DataFrame({"order": np.arange(1, len(B) + 1), "probability": _p["early"], "MACE": _yB})
p = np.clip(data["probability"].to_numpy(), 1e-6, 1 - 1e-6)
z = np.log(p / (1 - p))
y = data["MACE"].to_numpy()
first = data["order"].to_numpy() <= 300
a = 0.0
for _ in range(50):
    q = 1 / (1 + np.exp(-(a + z[first])))
    a += (y[first] - q).sum() / (q * (1 - q)).sum()
p_new = 1 / (1 + np.exp(-(z + a)))
rest = ~first
print("shift", round(a, 3), "| mean before", round(p[rest].mean(), 3), "| after", round(p_new[rest].mean(), 3),
      "| observed", round(y[rest].mean(), 3))
print("AUC", round(roc_auc_score(y[rest], p[rest]), 3), round(roc_auc_score(y[rest], p_new[rest]), 3))
answer = round(float(a), 2)
teach.check_step("Step 3.1", "answer", globals())

**What the result shows.** Hospital B learns a shift of 0.37: The model of A underestimates the risk at B, where the patients are older and angina is treated more invasively. The update moves every patient up by the same step on the logit scale and leaves the AUC unchanged.

### Step 3.2: A threshold for discharge

**Goal.** Choose on the validation presentations a threshold of the model that sends home as many patients as HEART 0 to 3, and count the MACE among the test patients it sends home.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether discharging by the risk of a model instead of by HEART 0 to 3, with the same number of patients sent home, would miss fewer major adverse cardiac events.  
> Data: The notebook holds a table named "data" with the 800 validation and the 800 test presentations of a chest pain unit of the virtual clinic, a synthetic data set. Each row is a presentation: "set" gives its set ("validation" or "test"), "HEART" the HEART score (0 to 10), "probability" the probability of a major adverse cardiac event within 30 days from a model of the course that uses five modalities, and "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: On the validation presentations, count the patients with HEART 0 to 3; call this number k. Sort the validation presentations by the probability; the threshold is the probability of the k-th lowest. On the test presentations, send home every patient at or below the threshold, and report the number sent home, the MACE among them, their share with a 95 percent Wilson interval, and the sensitivity for MACE. Report the same for HEART 0 to 3 on the test presentations.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the threshold and both results, and keep the number of MACE among the test patients sent home by the model under the name "answer".  
> Test: The threshold must come from the validation presentations only, and the number sent home by the model must be close to that of HEART.  

In [ ]:
#@title Data of step 3.2: run this cell (it prepares the table "data" and shows it)
_p = _course_probabilities()["A"]   # one to three minutes the first time
_k = _va | _te
data = pd.DataFrame({"set": np.where(_va, "validation", "test")[_k], "HEART": A["heart"].to_numpy()[_k],
                     "probability": _p["early"][_k], "MACE": _y[_k]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.2
teach.check_step("Step 3.2", "answer", globals())

In [ ]:
#@title Ready code of step 3.2 (run it only if the code of the assistant does not pass the check)
_p = _course_probabilities()["A"]   # one to three minutes the first time
_k = _va | _te
data = pd.DataFrame({"set": np.where(_va, "validation", "test")[_k], "HEART": A["heart"].to_numpy()[_k],
                     "probability": _p["early"][_k], "MACE": _y[_k]})


def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return c - h, c + h


va = data[data["set"] == "validation"]
te = data[data["set"] == "test"]
k = int((va["HEART"] <= 3).sum())
threshold = float(np.sort(va["probability"].to_numpy())[k - 1])
print("threshold", round(threshold, 4))
for name, home in (("model", te["probability"] <= threshold), ("HEART 0-3", te["HEART"] <= 3)):
    n_home, missed = int(home.sum()), int(te["MACE"][home].sum())
    lo, hi = wilson(missed, n_home)
    print(name, "| sent home", n_home, "| MACE", missed, "|", round(missed / n_home, 3), (round(lo, 3), round(hi, 3)),
          "| sensitivity", round(1 - missed / te["MACE"].sum(), 3))
answer = int(te["MACE"][te["probability"] <= threshold].sum())
teach.check_step("Step 3.2", "answer", globals())

**What the result shows.** With the same share sent home, the model misses 9 MACE among the test patients, about as many as HEART 0 to 3. For the decision to discharge, fusion is no better than the score in this unit, and 800 test patients could not show a small difference.

### Step 3.3: Net benefit for the decision to admit

**Goal.** Compute the net benefit of admitting by the risk of the model and by HEART 4 or more, at a threshold probability of 20 percent.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist would admit a patient with a risk of 20 percent of MACE, accepting four unnecessary admissions for every event caught, and wants to know whether the probabilities of a model serve this decision better than the rule "admit every patient with HEART 4 or more".  
> Data: The notebook holds a table named "data" with the 800 test presentations of a chest pain unit of the virtual clinic, a synthetic data set. Each row is a presentation: "HEART" gives the HEART score (0 to 10), "probability" the probability of a major adverse cardiac event within 30 days from a model of the course that uses five modalities, and "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: At the threshold probability t = 0.20, count the true positives (MACE among the admitted) and the false positives (no MACE among the admitted), once for admitting every patient whose probability is at least t and once for admitting every patient with HEART 4 or more. The net benefit is the true positives divided by the number of patients minus the false positives divided by the number of patients times t / (1 − t). Also give the net benefit of admitting everyone.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the counts and the net benefit of the three strategies, and keep the net benefit of the model, with three decimals, under the name "answer".  
> Test: The net benefit of admitting no one is 0, and that of the model must be computed on all 800 test presentations.  

In [ ]:
#@title Data of step 3.3: run this cell (it prepares the table "data" and shows it)
_p = _course_probabilities()["A"]   # one to three minutes the first time
data = pd.DataFrame({"HEART": A["heart"].to_numpy()[_te], "probability": _p["early"][_te], "MACE": _y[_te]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.3
teach.check_step("Step 3.3", "answer", globals())

In [ ]:
#@title Ready code of step 3.3 (run it only if the code of the assistant does not pass the check)
_p = _course_probabilities()["A"]   # one to three minutes the first time
data = pd.DataFrame({"HEART": A["heart"].to_numpy()[_te], "probability": _p["early"][_te], "MACE": _y[_te]})
t = 0.20
y = data["MACE"].to_numpy()
n = len(y)
answer = None
for name, admit in (("model", data["probability"].to_numpy() >= t), ("HEART ≥ 4", data["HEART"].to_numpy() >= 4),
                         ("everyone", np.ones(n, dtype=bool))):
    tp, fp = int((admit & (y == 1)).sum()), int((admit & (y == 0)).sum())
    nb = tp / n - fp / n * t / (1 - t)
    print(name, "| TP", tp, "| FP", fp, "| net benefit", round(nb, 3))
    if answer is None:
        answer = round(float(nb), 3)
teach.check_step("Step 3.3", "answer", globals())

**What the result shows.** At 20 percent the model gives a net benefit of 0.125, clearly above the rule of HEART 4 or more, which admits many patients without an event. For the decision to admit, probabilities serve better than three groups of points.

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#ai-pair-programming-part-8-reviewing-a-fusion-pipeline) and continue reading at «Working with an AI assistant, part 8: Reviewing a fusion pipeline». The weekly project can also be done on the prompt path: Its prompts are in the section «Weekly project on the prompt path» below.

## Weekly project on the prompt path

*Project of the week: [README of Week 8, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-08/README.md#weekly-project)*

The weekly project can also be built with the prompts below, one for each step. Run the preparation cell once: It loads the data of the project. Each step then works like a step of the stops, with one difference: The project has no check cell and no ready code, because its results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

In [ ]:
#@title Preparation of the project: run this cell once (it loads the data of the project; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
if "chest" not in globals():
    chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
    A = chest[chest["hospital"] == "A"].reset_index(drop=True)
    B = chest[chest["hospital"] == "B"].reset_index(drop=True)
    tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
    y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()
if "fit_logistic" not in globals():
    def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
        """Standardise the columns and fit a logistic regression; with a validation set, keep the inverse regularisation strength with the largest validation AUC."""
        best = None
        for C in (Cs if Xv is not None else (1.0,)):
            model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
            auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
            if best is None or auc > best[1]:
                best = (model, auc)
        return best[0]


    def logit(p):
        """The log-odds of a probability."""
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))


    def bootstrap_auc(y, p, n=2000, seed=2026):
        """AUC in n bootstrap samples of the patients: an array of n values."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p[i]))
        return np.array(out)


    def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
        """The AUC of p1 minus the AUC of p2 in n bootstrap samples of the same patients: an array of n differences."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
        return np.array(out)


    def calibration(y, p):
        """Calibration in the large (the intercept with the slope fixed at 1) and the calibration slope."""
        z = logit(p)
        a = 0.0
        for _ in range(50):
            q = 1 / (1 + np.exp(-(a + z)))
            a += (y - q).sum() / (q * (1 - q)).sum()
        slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
        return a, slope


    def reliability(y, p, bins=10):
        """Mean predicted risk and observed share of events in groups of equal size: two arrays."""
        order = np.argsort(p)
        groups = np.array_split(order, bins)
        return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])
_SETS = {"training": "training", "validation": "validation", "test": "test", "external": "hospital B"}
# sets and outcomes under private names, which the code of an assistant does not overwrite
_tr, _va, _te = [(A["set"] == _s).to_numpy() for _s in ("training", "validation", "test")]
_y, _yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()


def _unit_table(d):
    """The presentations of a hospital with what the team knows at arrival and MACE within 30 days."""
    url = d["troponin_url"]
    return pd.DataFrame({"presentation": d["patient_id"], "set": d["set"].map(_SETS), "age": d["age"],
                         "sex": d["sex"].map({"M": "man", "F": "woman"}),
                         "smoker": d["smoker"], "diabetes": d["diabetes"], "hypertension": d["hypertension"],
                         "hyperlipidaemia": d["hyperlipidaemia"], "obesity": d["obesity"],
                         "family history": d["family_history"], "known coronary artery disease": d["known_cad"],
                         "heart rate": d["heart_rate"], "systolic pressure": d["sbp"], "eGFR": d["egfr"],
                         "troponin at arrival": (d["troponin_0h"] / url).round(2),
                         "troponin after one hour": (d["troponin_1h"] / url).round(2),
                         "echocardiogram done": d["echo_done"], "ejection fraction": d["lvef"],
                         "wall motion abnormality": d["rwma"], "HEART": d["heart"], "MACE": d["mace30"]})


def _course_probabilities():
    """The probabilities of MACE that the models of the course give at hospitals A and B: five single modalities, early fusion and the mean of the logits (computed once, one to three minutes, most of it for the network of the ECG)."""
    if "_prob_cache" in globals():
        return _prob_cache
    tr, va, y = _tr, _va, _y
    CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
                "heart_rate", "sbp", "egfr"]


    def clinical(d):
        """The clinical table: age, sex, risk factors, known coronary artery disease, heart rate, systolic pressure and eGFR."""
        return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


    def laboratory(d):
        """Troponin in multiples of the upper reference limit of its assay: the log of the value at arrival and after one hour, the change in one hour, and whether the second value exists."""
        url = d["troponin_url"]
        t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
        return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                             "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


    def echo(d):
        """The bedside echocardiogram: whether it was done, the ejection fraction (60 when not done) and a regional wall motion abnormality (0 when not done)."""
        return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
    svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


    def note(d):
        """The triage note as 20 numbers: TF-IDF of words and word pairs, reduced by a singular value decomposition fitted on the training notes."""
        return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

    ecg_A = triage.ecg_array(A)          # 4,000 ECGs of 12 leads, 10 seconds at 100 Hz: 4000 x 12 x 1000
    ecg_B = triage.ecg_array(B)


    class ECGNet(nn.Module):
        """A one-dimensional convolutional network for a 12-lead ECG: four blocks, the mean of every channel over time and one output."""
        def __init__(self):
            super().__init__()
            block = lambda a, b: nn.Sequential(nn.Conv1d(a, b, 7, padding=3), nn.BatchNorm1d(b), nn.ReLU(), nn.MaxPool1d(2))
            self.features = nn.Sequential(block(12, 32), block(32, 64), block(64, 64), block(64, 64))
            self.head = nn.Linear(64, 1)

        def embed(self, x):
            return self.features(x).mean(dim=2)

        def forward(self, x):
            return self.head(self.embed(x)).squeeze(1)


    def train_ecg(model, X, y, Xv, yv, epochs=12, seed=2026, verbose=True):
        """Adam with a learning rate of 0.001 in groups of 64 ECGs, each shifted in time by up to half a second; the weights of the epoch with the best validation AUC are kept."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.roll(X[b], int(torch.randint(-50, 51, (1,))), dims=2)
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, ecg_predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    @torch.no_grad()
    def ecg_predict(model, X, batch=256):
        model.eval()
        return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


    @torch.no_grad()
    def ecg_embed(model, X, batch=256):
        model.eval()
        return torch.cat([model.embed(torch.as_tensor(X[i:i + batch])) for i in range(0, len(X), batch)]).numpy()
    torch.manual_seed(RANDOM_STATE)
    net = ECGNet()
    train_ecg(net, ecg_A[tr], y[tr], ecg_A[va], y[va], verbose=False)
    emb = lambda X: pd.DataFrame(ecg_embed(net, X), columns=[f"ecg_{i + 1}" for i in range(64)])
    out = {}
    for name, d, X in (("A", A, ecg_A), ("B", B, ecg_B)):
        parts = {"clinical": clinical(d), "troponin": laboratory(d), "echocardiogram": echo(d), "note": note(d)}
        out[name] = dict(parts=parts, ecg=ecg_predict(net, X), embedding=emb(X))
    p = {"A": {}, "B": {}}
    for k in out["A"]["parts"]:
        m = fit_logistic(out["A"]["parts"][k][tr], y[tr], Xv=out["A"]["parts"][k][va], yv=y[va])
        for h in ("A", "B"):
            p[h][k] = m.predict_proba(out[h]["parts"][k])[:, 1]
    for h in ("A", "B"):
        p[h]["ECG network"] = out[h]["ecg"]
    early = lambda h: pd.concat([out[h]["parts"]["clinical"], out[h]["parts"]["troponin"],
                                 out[h]["parts"]["echocardiogram"], out[h]["embedding"], out[h]["parts"]["note"]], axis=1)
    m = fit_logistic(early("A")[tr], y[tr], Xv=early("A")[va], yv=y[va])
    for h in ("A", "B"):
        p[h]["early"] = m.predict_proba(early(h))[:, 1]
        z = np.column_stack([logit(p[h][k]) for k in ["clinical", "troponin", "echocardiogram", "ECG network", "note"]])
        p[h]["mean"] = 1 / (1 + np.exp(-z.mean(axis=1)))
    globals()["_prob_cache"] = p
    return p


print(len(A), "presentations at hospital A and", len(B), "at hospital B")



def _project_unit():
    """The presentations of hospital A for the first step: the unit table with the scores of the course models for the ECG and the note."""
    p = _course_probabilities()["A"]
    t = _unit_table(A)
    t["ECG score"] = logit(p["ECG network"]).round(3)
    t["note score"] = logit(p["note"]).round(3)
    return t


def _project_both():
    """The validation and test presentations of hospital A and the presentations of hospital B with the probability of the course model, the HEART score and MACE."""
    p = _course_probabilities()
    k = _va | _te
    a = pd.DataFrame({"set": np.where(_va, "validation", "test")[k], "order": 0, "HEART": A["heart"].to_numpy()[k],
                      "probability": p["A"]["early"][k], "MACE": _y[k]})
    b = pd.DataFrame({"set": "hospital B", "order": np.arange(1, len(B) + 1), "HEART": B["heart"].to_numpy(),
                      "probability": p["B"]["early"], "MACE": _yB})
    return pd.concat([a, b], ignore_index=True)
print("The project is ready.")

### Project step 1: Single modalities, fusion and HEART

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compare models for MACE within 30 days from each modality of a chest pain unit and from all of them together with the HEART score, with intervals that show how sure each difference is.  
> Data: The notebook holds a table named "data" with the 4,000 presentations with chest pain of a chest pain unit of the virtual clinic, a synthetic data set. Each row is a presentation: "presentation" gives its number and "set" its set ("training", 2,400; "validation", 800; "test", 800). The clinical table gives "age" (years), "sex" ("man" or "woman"), seven risk factors that are 1 when present ("smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family history" and "known coronary artery disease"), "heart rate" (per minute), "systolic pressure" (mmHg) and "eGFR". High-sensitivity troponin T is given in multiples of the upper reference limit of the assay: "troponin at arrival" and "troponin after one hour", which is empty when it was not measured. The bedside echocardiogram gives "echocardiogram done" (1 or 0), and when it was done "ejection fraction" (%) and "wall motion abnormality" (1 when a regional abnormality was seen); both are empty otherwise. "HEART" is the HEART score, and "MACE" is 1 when a major adverse cardiac event followed within 30 days and 0 otherwise. Two more columns come from models of the course trained on the training presentations: "ECG score", the logit of the probability of MACE that a network gives from the 12-lead ECG, and "note score", the logit of the probability from the triage note.  
> Task: Build five groups of features: the clinical table (age, sex as 1 for a man, the seven risk factors, heart rate, systolic pressure, eGFR); the troponin (the logarithm of the troponin at arrival, the logarithm of the troponin after one hour with the value at arrival when it was not measured, the change in one hour with 0 when not measured, and a column that is 1 when the second troponin was measured); the echocardiogram (whether it was done, the ejection fraction with 60 and the wall motion abnormality with 0 when not done); the ECG score; and the note score. Fit a model on each group alone and one on all groups together (early fusion), each as follows. Standardise the columns with the training presentations and fit a logistic regression on the training presentations. Choose the amount of regularisation by the area under the ROC curve (AUC) on the validation presentations, trying 0.01, 0.1, 1 and 10 for the inverse of its strength, and compute the AUC of the chosen model on the test presentations. Give every AUC a 95 percent interval from 2,000 bootstrap samples of the test presentations, from the 2.5th to the 97.5th percentile, and compute the difference of each model to the HEART score in the same samples. Fix the random seed to 2026 so that the result can be repeated.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each model and for HEART: its test AUC with the interval and its difference to HEART with the interval; keep the test AUC of early fusion, with three decimals, under the name "answer".  
> Test: No test presentation may be used for a choice, and the HEART score must not be a feature of any model.  

In [ ]:
#@title Data of project step 1: run this cell (it prepares the table "data" and shows it)
data = _project_unit()   # one to three minutes the first time
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: Calibration at two hospitals

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether the probabilities of a risk model for MACE can be trusted on the test patients of the hospital where it learned and at a second hospital, before and after an update with its first patients.  
> Data: The notebook holds a table named "data" with presentations with chest pain of two hospitals of the virtual clinic, a synthetic data set: the 800 validation and the 800 test presentations of hospital A and the 1,000 presentations of hospital B. Each row is a presentation: "set" gives "validation", "test" or "hospital B", "order" the place in time of a presentation of B (1 to 1,000; 0 for A), "HEART" the HEART score (0 to 10), "probability" the probability of a major adverse cardiac event within 30 days from a model of hospital A that fuses five modalities, and "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: For the test presentations of A, compute the mean predicted risk against the share of MACE. Compute the calibration intercept (the shift of the logits that makes the mean predicted risk equal to the observed share, with the slope fixed at 1), the calibration slope (a logistic regression of MACE on the logit of the probability, without regularisation) and the Brier score. Draw a reliability curve of ten groups of equal size with the diagonal. For hospital B, find on the presentations with order 1 to 300 the shift of the logits that makes their mean predicted risk equal to their share of MACE, add it to the logits of all presentations of B, and repeat the measures and the curve for the presentations with order 301 to 1,000, before and after the update.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the measures in a table with three rows (A test, B before, B after) and the three curves; keep the calibration intercept at B after the update, with two decimals, under the name "answer".  
> Test: Only the presentations of B with order 1 to 300 may be used to find the shift.  

In [ ]:
#@title Data of project step 2: run this cell (it prepares the table "data" and shows it)
data = _project_both()   # one to three minutes the first time
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: Two thresholds for discharge

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to choose a threshold for discharging patients with chest pain by the risk of a model, and to know how many events it would miss at the hospital where it was chosen and at another hospital, next to HEART 0 to 3.  
> Data: The notebook holds a table named "data" with presentations with chest pain of two hospitals of the virtual clinic, a synthetic data set: the 800 validation and the 800 test presentations of hospital A and the 1,000 presentations of hospital B. Each row is a presentation: "set" gives "validation", "test" or "hospital B", "order" the place in time of a presentation of B (1 to 1,000; 0 for A), "HEART" the HEART score (0 to 10), "probability" the probability of a major adverse cardiac event within 30 days from a model of hospital A that fuses five modalities, and "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: Choose two thresholds on the validation presentations of A only. First: count the validation presentations with HEART 0 to 3, k, and take the k-th lowest probability. Second: the largest probability at which at most 2 percent of the validation presentations with MACE fall at or below it. For each threshold and for HEART 0 to 3, report on the test presentations of A and on the presentations of B with order 301 to 1,000: the number and the share sent home (at or below the threshold), the MACE among them with a 95 percent Wilson interval, and the sensitivity for MACE.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each rule at each hospital, and keep the number of MACE among the test presentations of A sent home at the first threshold under the name "answer".  
> Test: Both thresholds must come from the validation presentations of A only.  

In [ ]:
#@title Data of project step 3: run this cell (it prepares the table "data" and shows it)
data = _project_both()   # one to three minutes the first time
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: A decision curve for the heart team

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to show a heart team for which decision a risk model for MACE helps, by comparing its net benefit with that of the HEART score over a range of thresholds for admission.  
> Data: The notebook holds a table named "data" with presentations with chest pain of two hospitals of the virtual clinic, a synthetic data set: the 800 validation and the 800 test presentations of hospital A and the 1,000 presentations of hospital B. Each row is a presentation: "set" gives "validation", "test" or "hospital B", "order" the place in time of a presentation of B (1 to 1,000; 0 for A), "HEART" the HEART score (0 to 10), "probability" the probability of a major adverse cardiac event within 30 days from a model of hospital A that fuses five modalities, and "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: Use the test presentations of A. For every threshold probability t from 0.01 to 0.30 in steps of 0.01, compute the net benefit of admitting every presentation whose probability is at least t, of admitting every presentation with HEART 4 or more, and of admitting everyone: the true positives divided by the number of presentations minus the false positives divided by the number of presentations times t / (1 − t). Draw the three curves with the line at 0 for admitting no one.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the decision curve and print the three net benefits at 0.05, 0.10, 0.20 and 0.30; keep the net benefit of the model at 0.20, with three decimals, under the name "answer".  
> Test: The net benefit of admitting no one is 0 at every threshold.  

In [ ]:
#@title Data of project step 4: run this cell (it prepares the table "data" and shows it)
data = _project_both()   # one to three minutes the first time
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

<a id="part-2"></a>

# Part 2: Python path

**For whom?** This part is for participants who want to learn to write Python. Part 1 is not needed for it: After Section 0, start here.

**How it works.** Run the cells from top to bottom with Shift+Enter. The *Python step* at the start teaches the parts of the language that the week needs. In the numbered sections, exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. Cells marked *Advanced (optional)* are extensions for those who already program. A participant who opens this part in the middle runs the *Catch-up* cell at the start of that section. Some exercise cells begin with a line that prepares a table named `data`: It serves the prompt of the exercise in the prompt workshop of the interactive lab, and on the Python path the exercise can be solved with the tables named in its text.

## Python step 8: Joining data

This week the data of one patient come from several tables and arrays, and they must meet in the right row. This step shows how two tables are joined by the number of the patient with `merge`, how an array of signals stays aligned with its table, how a mask of the table selects the same patients in the array, and how missing values are found, filled and marked. Every example uses the chest pain unit of the virtual clinic.

In [ ]:
#@title Preparation of the Python step: run this cell once (if Section 0 has not run in this session, it runs it)
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
print("The Python step is ready.")

In [ ]:
# Python practice: a small example for learning the language
unit = triage.presentations("A")
clinical_part = unit[["patient_id", "age", "sex", "heart_rate", "sbp"]]
lab_part = unit[["patient_id", "troponin_0h", "troponin_1h", "troponin_url"]].sample(frac=1, random_state=RANDOM_STATE)
print("first rows:", clinical_part["patient_id"].head(3).tolist(), lab_part["patient_id"].head(3).tolist())
joined = clinical_part.merge(lab_part, on="patient_id", how="left")
print(clinical_part.shape, "+", lab_part.shape, "→", joined.shape)
print(joined[joined["patient_id"] == "CP-3304"])

**Python note.** `triage.presentations("A")` draws the 4,000 presentations of hospital A, one row each. Two parts of the table are cut out with double brackets, a list of column names, and the laboratory part is shuffled with `sample(frac=1)`, as if it came from another system in another order. `merge(lab_part, on="patient_id", how="left")` puts the columns of the second table next to those of the first, row by row, wherever the patient numbers match; *left* keeps every row of the first table even when the second has no match, with an empty value in that case. Never join two tables by their order: Only a shared number of the patient makes sure that the values of one patient meet.

In [ ]:
# Python practice: a signal and its row
first = unit.head(5)
signals = triage.ecg_array(first)
print("signals:", signals.shape)
row = 3
print(first["patient_id"].iloc[row], first["ecg_reading"].iloc[row], signals[row].shape)
normal = (first["ecg_reading"] == "normal").to_numpy()
print(normal, "→", signals[normal].shape)

**Python note.** `triage.ecg_array` draws the 12-lead ECG of every row of a table and returns one array of 5 × 12 × 1000: 5 presentations, 12 leads and 10 seconds at 100 samples per second. Row 3 of the array belongs to row 3 of the table because both were made in the same order, and they must stay in that order: Sorting or filtering the table without doing the same to the array would give the ECG of one patient to another. A mask, an array of `True` and `False` made from the table with `.to_numpy()`, selects the same patients in both: `first[normal]` and `signals[normal]`.

In [ ]:
# Python practice: missing parts
print(unit[["troponin_1h", "lvef", "rwma"]].isna().mean().round(3))
filled = unit["lvef"].fillna(60.0)
done = unit["lvef"].notna().astype(int)
print(pd.DataFrame({"lvef": unit["lvef"], "filled": filled, "done": done}).head(6))
print(pd.concat([unit[["age"]], done.rename("echo_done"), filled.rename("lvef")], axis=1).shape)

**Python note.** `isna()` gives `True` for every empty cell, and the mean of `True` and `False` is the share of empty cells in each column. `fillna(60.0)` puts a typical ejection fraction into the gaps, and `notna().astype(int)` keeps, in a column of its own, the fact that a value was there: 1 when the echocardiogram was done, 0 when it was not. A model then sees both the filled value and the gap. `pd.concat([...], axis=1)` puts tables and columns with the same rows side by side; `axis=0`, the default, would put them under each other.

**Quick check.** A table `X` holds 4,000 patients and a table `Y` 3,000 of them. How many rows does `X.merge(Y, on="patient_id", how="left")` have?

<details><summary>Answer</summary>

4,000: A left join keeps every row of `X`; the 1,000 patients without a row in `Y` get empty values in its columns.

</details>

**Exercise 8.1.** What share of the presentations of hospital A had a second troponin measured? Keep it with three decimals.

In [ ]:
data = exercise_table("8.1")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 8.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(unit["troponin_1h"].notna().mean()), 3)
teach.check("Exercise 8.1", answer)

**Exercise 8.2.** Among the presentations with an echocardiogram (`echo_done` equal to 1), what share had a MACE (`mace30`)? Keep it with three decimals.

In [ ]:
data = exercise_table("8.2")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 8.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(unit.loc[unit["echo_done"] == 1, "mace30"].mean()), 3)
teach.check("Exercise 8.2", answer)

**Exercise 8.3.** How many presentations of hospital A have both a second troponin and an echocardiogram? Keep the number.

In [ ]:
data = exercise_table("8.3")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 8.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((unit["troponin_1h"].notna() & (unit["echo_done"] == 1)).sum())
teach.check("Exercise 8.3", answer)

## 1. The registry and three modalities

*Lecture: [Mini project: Heart failure risk classification with a fusion model of the ECG signal, echocardiography and laboratory values](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#mini-project-heart-failure-risk-classification-with-a-fusion-model-of-the-ecg-signal-echocardiography-and-laboratory-values)*

The mini project returns to the registry of the virtual clinic with its follow-up. The outcome is death or a hospitalisation for heart failure within 24 months; patients followed for less than 24 months without an event are left out, as Week 2 showed. Three modalities describe each patient: the ECG, the echocardiogram and the laboratory values. This section fits one logistic regression for each modality alone.

**From the formulas to the code.** The numbers of modality $m$, $x_m$: `registry[HF_GROUPS[name]]`. A model of one modality and its probabilities $p_m$: `fit_logistic(...)` and `probs_hf[name]`. The regularisation is chosen on the validation patients.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(1, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
registry = followup.add_followup(cohort.make_cohort(3600))
# an event within 24 months; patients followed for less than 24 months without an event are left out
registry = registry[(registry["event"] == 1) | (registry["followup_months"] >= 24)].reset_index(drop=True)
registry["event24"] = ((registry["event"] == 1) & (registry["followup_months"] <= 24)).astype(int)
number = registry["patient_id"].str[3:].astype(int)
registry["set"] = np.select([number <= 2160, number <= 2880], ["training", "validation"], "test")
registry["lbbb"] = (registry["ecg_conduction"] == "LBBB").astype(int)
registry["af_ecg"] = (registry["ecg_rhythm"] == "AF").astype(int)
registry["log_ntprobnp"] = np.log(registry["ntprobnp"])
registry["log_troponin"] = np.log(registry["troponin"])
HF_GROUPS = {
    "ECG": ["ecg_hr", "ecg_qrs", "ecg_qt", "lbbb", "af_ecg"],
    "echo": ["lvef", "lvedd_mm", "lavi", "e_e_prime", "tapse_mm", "lv_mass_index"],
    "laboratory": ["log_ntprobnp", "egfr", "log_troponin"],
}

print(len(registry), "patients,", int(registry["event24"].sum()), "with an event within 24 months")
print(registry.groupby("set")["event24"].agg(["size", "sum"]))
for name, cols in HF_GROUPS.items():
    print(f"{name:14}", cols)

**Python note.** `followup.add_followup(cohort.make_cohort(3600))` draws the 3,600 patients of the registry with the follow-up of Week 2: the months followed, whether an event happened and which. The line with `|` keeps the patients who had an event or were followed for at least 24 months; the others are left out, since nobody knows what happened to them by month 24. `str[3:]` cuts the number out of a patient number such as VC-0317, and `np.select` turns it into a set: up to 2,160 training, up to 2,880 validation, the rest test. `HF_GROUPS` is a dictionary from the name of a modality to the list of its columns; NT-proBNP and troponin enter as logarithms, since both are strongly skewed.

In [ ]:
def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
    """Standardise the columns and fit a logistic regression; with a validation set, keep the inverse regularisation strength with the largest validation AUC."""
    best = None
    for C in (Cs if Xv is not None else (1.0,)):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
        auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
        if best is None or auc > best[1]:
            best = (model, auc)
    return best[0]


def logit(p):
    """The log-odds of a probability."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def bootstrap_auc(y, p, n=2000, seed=2026):
    """AUC in n bootstrap samples of the patients: an array of n values."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p[i]))
    return np.array(out)


def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
    """The AUC of p1 minus the AUC of p2 in n bootstrap samples of the same patients: an array of n differences."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    return np.array(out)


def calibration(y, p):
    """Calibration in the large (the intercept with the slope fixed at 1) and the calibration slope."""
    z = logit(p)
    a = 0.0
    for _ in range(50):
        q = 1 / (1 + np.exp(-(a + z)))
        a += (y - q).sum() / (q * (1 - q)).sum()
    slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
    return a, slope


def reliability(y, p, bins=10):
    """Mean predicted risk and observed share of events in groups of equal size: two arrays."""
    order = np.argsort(p)
    groups = np.array_split(order, bins)
    return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])

sets_r = registry["set"].to_numpy()
rtr, rva, rte = sets_r == "training", sets_r == "validation", sets_r == "test"
yr = registry["event24"].to_numpy()
probs_hf = {}
for name, cols in HF_GROUPS.items():
    model = fit_logistic(registry.loc[rtr, cols], yr[rtr], Xv=registry.loc[rva, cols], yv=yr[rva])
    probs_hf[name] = model.predict_proba(registry[cols])[:, 1]
    print(f"{name:14} validation AUC {roc_auc_score(yr[rva], probs_hf[name][rva]):.3f}   test AUC {roc_auc_score(yr[rte], probs_hf[name][rte]):.3f}")

**Python note.** `fit_logistic` standardises the columns and fits a logistic regression for each of four strengths of regularisation, `Cs`, keeping the model with the largest AUC on the validation patients; the test patients are never used for a choice. `registry.loc[rtr, cols]` takes the rows of the training patients and the columns of one modality. `predict_proba(...)[:, 1]` gives the probability of an event for every patient of the registry, so the same array serves the three sets.

**Observe and reflect.** Which modality alone predicts heart failure events best, and what does NT-proBNP measure that the other two modalities show only indirectly?

**Exercise A.** How many test patients of the registry had an event within 24 months? Keep the number.

In [ ]:
data = exercise_table("A")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(yr[rte].sum())
teach.check("Exercise A", answer)

## 2. Early and late fusion

*Lecture: [Mini project: Heart failure risk classification with a fusion model of the ECG signal, echocardiography and laboratory values](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#mini-project-heart-failure-risk-classification-with-a-fusion-model-of-the-ecg-signal-echocardiography-and-laboratory-values)*

Early fusion fits one model on the 14 values of the three modalities side by side. Late fusion combines the three models of Section 1: by the mean of their logits, an equal vote, or by stacking, a second logistic regression that learns on the validation patients how much each model counts.

**From the formulas to the code.** $[\,x_1; x_2; x_3\,]$: `registry[cols_all]`. $z_m = \ln(p_m/(1 - p_m))$: `logit(probs_hf[name])`, one column of `logits_hf`. The mean $\frac{1}{M}\sum_m z_m$: `logits_hf.mean(axis=1)`. Stacking $b + \sum_m w_m z_m$: `stack_hf`, with $w_m$ in `stack_hf.coef_` and $b$ in `stack_hf.intercept_`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(2, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
cols_all = sum(HF_GROUPS.values(), [])
early_hf = fit_logistic(registry.loc[rtr, cols_all], yr[rtr], Xv=registry.loc[rva, cols_all], yv=yr[rva])
probs_hf["early"] = early_hf.predict_proba(registry[cols_all])[:, 1]
logits_hf = np.column_stack([logit(probs_hf[name]) for name in HF_GROUPS])
probs_hf["late"] = 1 / (1 + np.exp(-logits_hf.mean(axis=1)))
stack_hf = LogisticRegression(max_iter=5000).fit(logits_hf[rva], yr[rva])
probs_hf["stacked"] = stack_hf.predict_proba(logits_hf)[:, 1]
print("stacking weights:", dict(zip(HF_GROUPS, np.round(stack_hf.coef_[0], 2))), "| intercept", round(stack_hf.intercept_[0], 2))
auc_hf = {name: roc_auc_score(yr[rte], p[rte]) for name, p in probs_hf.items()}
fig, ax = plt.subplots(figsize=(6.4, 3))
ax.barh(list(auc_hf), list(auc_hf.values()), color=["#8A8A8A"] * 3 + ["#1F5F8B", "#0E7A78", "#6E46A0"])
for i, v in enumerate(auc_hf.values()):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("test AUC")
ax.invert_yaxis()
plt.show()

**Python note.** `sum(HF_GROUPS.values(), [])` adds the three lists of columns into one list of 14, the row of early fusion. `np.column_stack` puts the three arrays of logits side by side as the three columns of one array, one row per patient. The mean of the logits needs no fitting. Stacking is fitted on the validation patients only, because the three models were fitted on the training patients: On their own training data their probabilities are too good, and a stacker fitted there would trust them too much. `LogisticRegression(max_iter=5000)` keeps its default regularisation.

**Observe and reflect.** Early fusion uses 14 values and the laboratory 3. Why might more values give no better test AUC, and what does the weight of the ECG in stacking suggest?

**Exercise B.** What is the test AUC of the late fusion of the echocardiogram and the laboratory only, by the mean of their two logits? Keep it with three decimals.

In [ ]:
data = exercise_table("B")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
z = (logit(probs_hf["echo"]) + logit(probs_hf["laboratory"])) / 2
answer = round(float(roc_auc_score(yr[rte], z[rte])), 3)
teach.check("Exercise B", answer)

## 3. How sure is the difference?

*Lecture: [Intervals: How sure is a difference?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#intervals-how-sure-is-a-difference)*

The test set holds a few dozen events. A bootstrap draws the test patients again and again with replacement, and in every sample both models are judged on the same patients, so that the difference between their AUCs carries an interval.

**From the formulas to the code.** The difference in one sample: `roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i])`. The 2,000 differences: `paired_bootstrap(y, p1, p2)`. The 95 percent interval: `np.percentile(d, [2.5, 97.5])`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(3, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
diffs_hf = {}
for name in list(HF_GROUPS) + ["late", "stacked"]:
    d = paired_bootstrap(yr[rte], probs_hf["early"][rte], probs_hf[name][rte])
    diffs_hf[name] = d
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"early fusion minus {name:14} {d.mean():+.3f}  ({lo:+.3f} to {hi:+.3f})  share above 0 {(d > 0).mean():.3f}")
fig, ax = plt.subplots(figsize=(6.4, 2.8))
for i, (name, d) in enumerate(diffs_hf.items()):
    lo, hi = np.percentile(d, [2.5, 97.5])
    ax.plot([lo, hi], [i, i], color="#1F5F8B", lw=2)
    ax.plot(d.mean(), i, "o", color="#1F5F8B")
ax.axvline(0, color="grey", ls="--")
ax.set_yticks(range(len(diffs_hf)))
ax.set_yticklabels(list(diffs_hf))
ax.set_xlabel("AUC of early fusion minus AUC of the model")
ax.invert_yaxis()
plt.show()

**Python note.** `paired_bootstrap` draws as many test patients as there are, with replacement, with `rng.integers`; a sample without an event or without a patient free of events cannot give an AUC and is skipped by the `while` loop. Both models are scored on the same drawn patients, which keeps their shared luck out of the difference. The seed 2026 of `np.random.default_rng` makes every run draw the same samples. The line of each model in the plot is the 95 percent interval of the difference; where it crosses the dashed line at 0, the data do not decide.

**Observe and reflect.** With 34 test events, how large would a difference in AUC have to be before its interval no longer crossed 0? What would you tell a colleague who reports that fusion is better because its AUC is higher?

**Exercise C.** In what share of the 2,000 bootstrap samples of the test patients does the echocardiogram alone reach a higher AUC than the ECG alone? Keep it with three decimals.

In [ ]:
data = exercise_table("C")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float((paired_bootstrap(yr[rte], probs_hf["echo"][rte], probs_hf["ECG"][rte]) > 0).mean()), 3)
teach.check("Exercise C", answer)

**The Python path of Practice stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#last-two-hours-cardiovascular-risk-scoring-system) and continue reading at «Last two hours: Cardiovascular risk scoring system».

## 4. The chest pain unit and the HEART score

*Lecture: [Early warning model for suspected ACS](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#a-risk-model-for-triage-heart-and-the-fused-models)*

Hospital A, the chest pain unit of the course, holds 4,000 presentations with chest pain, split by their number into 2,400 for training, 800 for validation and 800 for the test; hospital B holds 1,000. The outcome is a major adverse cardiac event (MACE) within 30 days. The HEART score of every presentation is the bar for the models.

**From the formulas to the code.** The five items of HEART, 0 to 2 points each: `heart_h`, `heart_e`, `heart_a`, `heart_r` and `heart_t`; their sum: `heart`. The groups 0 to 3, 4 to 6 and 7 to 10: `pd.cut(..., [-1, 3, 6, 10])`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(4, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
A = chest[chest["hospital"] == "A"].reset_index(drop=True)
B = chest[chest["hospital"] == "B"].reset_index(drop=True)
tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()

print(chest.groupby(["hospital", "set"])["mace30"].agg(["size", "sum", "mean"]).round(3))
print(A.loc[A["patient_id"] == "CP-3304", ["age", "sex", "hours", "ecg_reading", "troponin_0h", "troponin_1h", "troponin_url", "lvef", "rwma", "heart", "diagnosis", "mace_type"]].T)
print(A.loc[A["patient_id"] == "CP-3304", "note_en"].iloc[0])

**Python note.** `triage.presentations("A")` and `("B")` draw the two hospitals, and `pd.concat(..., ignore_index=True)` puts them under each other in one table, `chest`, numbered from 0. The two hospitals are then taken out again as `A` and `B`, each with a fresh numbering from 0, so that the rows of the tables and of every array made from them stay aligned. `tr`, `va` and `te` are masks of the sets of hospital A, and `y` and `yB` the outcomes as arrays. `groupby(["hospital", "set"])` counts the presentations and the events of every set.

In [ ]:
heart = A["heart"].to_numpy()
group = pd.cut(A["heart"], [-1, 3, 6, 10], labels=["0-3", "4-6", "7-10"])
print(A[te].groupby(group[te], observed=True)["mace30"].agg(["size", "sum", "mean"]).round(3))
auc_heart = roc_auc_score(y[te], heart[te])
print("HEART, test AUC", round(auc_heart, 3), "| 95% interval", np.round(np.percentile(bootstrap_auc(y[te], heart[te].astype(float)), [2.5, 97.5]), 3))
fig, ax = plt.subplots(figsize=(6, 2.8))
share = A[te].groupby(A.loc[te, "heart"])["mace30"].mean()
ax.bar(share.index, share.values, color=["#0E7A78"] * 4 + ["#B97813"] * 3 + ["#9B1B1B"] * 4)
ax.set_xlabel("HEART score")
ax.set_ylabel("share with MACE")
plt.show()

**Python note.** `pd.cut` sorts every score into the interval it falls in: (−1, 3], (3, 6] and (6, 10], labelled 0-3, 4-6 and 7-10. `groupby(group[te], observed=True)` groups the test presentations by these labels and `agg(["size", "sum", "mean"])` gives the number of presentations, the number of events and their share in each group. An integer score can enter `roc_auc_score` directly: The AUC uses only the order of the patients. `bootstrap_auc` of Section 1 gives its interval.

**Observe and reflect.** How steeply does the share of MACE rise with the HEART score? Where would you put the line for discharge, and how many events would you accept below it?

**Exercise D.** What share of the test presentations with HEART 0 to 3 had a MACE? Keep it with three decimals.

In [ ]:
data = exercise_table("D")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(y[te][heart[te] <= 3].mean()), 3)
teach.check("Exercise D", answer)

## 5. Five modalities, five models

*Lecture: [The data of a heart team: The chest pain unit of the virtual clinic](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#the-data-of-a-heart-team-the-chest-pain-unit-of-the-virtual-clinic)*

Every modality becomes a table of numbers: the clinical table (12 values), the troponin in multiples of the upper reference limit of its assay (4 values), the echocardiogram with markers for a missing study (3 values) and the triage note in Turkish as TF-IDF of words and word pairs reduced to 20 numbers. A logistic regression is fitted on each of them. The ECG follows in Section 6.

**From the formulas to the code.** $x_m$ of each modality: `clinical(d)`, `laboratory(d)`, `echo(d)` and `note(d)`. The probabilities $p_m$ at both hospitals: `P[name]` and `PB[name]`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(5, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
            "heart_rate", "sbp", "egfr"]


def clinical(d):
    """The clinical table: age, sex, risk factors, known coronary artery disease, heart rate, systolic pressure and eGFR."""
    return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


def laboratory(d):
    """Troponin in multiples of the upper reference limit of its assay: the log of the value at arrival and after one hour, the change in one hour, and whether the second value exists."""
    url = d["troponin_url"]
    t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
    return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                         "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


def echo(d):
    """The bedside echocardiogram: whether it was done, the ejection fraction (60 when not done) and a regional wall motion abnormality (0 when not done)."""
    return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


def note(d):
    """The triage note as 20 numbers: TF-IDF of words and word pairs, reduced by a singular value decomposition fitted on the training notes."""
    return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

tables = {"clinical": (clinical(A), clinical(B)), "troponin": (laboratory(A), laboratory(B)),
          "echocardiogram": (echo(A), echo(B)), "note": (note(A), note(B))}
P, PB = {}, {}
for name, (X, XB) in tables.items():
    model = fit_logistic(X[tr], y[tr], Xv=X[va], yv=y[va])
    P[name], PB[name] = model.predict_proba(X)[:, 1], model.predict_proba(XB)[:, 1]
    print(f"{name:16} {X.shape[1]:3} values   test AUC {roc_auc_score(y[te], P[name][te]):.3f}   hospital B {roc_auc_score(yB, PB[name]):.3f}")

**Python note.** Each of the four functions takes a table of presentations and returns the numbers of one modality, one row per presentation in the same order: `clinical(A)` for hospital A and `clinical(B)` for hospital B. `laboratory` divides each troponin by the upper reference limit of its own assay, so the values of A (troponin T) and B (troponin I) mean the same thing. The TF-IDF and the singular value decomposition of `note` are fitted on the notes of the training presentations only, then applied to all notes. The dictionary `tables` holds the pair of tables of each modality, and the loop fits one model for each.

In [ ]:
words = tfidf.get_feature_names_out()
weights = LogisticRegression(max_iter=5000).fit(tfidf.transform(A.loc[tr, "note_tr"]), y[tr]).coef_[0]
order = np.argsort(weights)
print("words that lower the risk:", list(words[order[:8]]))
print("words that raise the risk:", list(words[order[-8:]]))

**Python note.** `get_feature_names_out` gives the words and word pairs that TF-IDF keeps. A plain logistic regression on all of them, without the reduction to 20 numbers, shows which terms push the risk up or down: `np.argsort` orders the weights, the first ones are the most negative and the last ones the most positive. The notes are written in Turkish, so the terms are Turkish in both versions of the notebook. Negations matter: "yoktur" (is absent) and "değildir" (is not) turn a feature around.

**Observe and reflect.** The note alone reaches about the same AUC as the clinical table. Which words carry the most weight, and why does the note lose more than any other modality at hospital B, whose notes are short?

**Exercise E.** How many words and word pairs does the TF-IDF of the training notes keep, `tfidf.vocabulary_`? Keep the number.

In [ ]:
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = len(tfidf.vocabulary_)
teach.check("Exercise E", answer)

## 6. The ECG with a one-dimensional convolutional network

*Lecture: [The data of a heart team: The chest pain unit of the virtual clinic](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#the-data-of-a-heart-team-the-chest-pain-unit-of-the-virtual-clinic)*

The 12-lead ECG of every presentation, 10 seconds at 100 Hz, enters a one-dimensional convolutional network like that of Week 5: four blocks of convolutions over time, the mean of every channel over time, and one output. The 64 numbers before the output describe the tracing and become the ECG modality of the fusion models in Section 7.

**From the formulas to the code.** The tracings: `ecg_A` and `ecg_B`, arrays of presentations × 12 leads × 1,000 samples. The probability $p_{\mathrm{ECG}}$: `ecg_predict(ecgnet, ...)`; the 64 numbers: `ecg_embed(ecgnet, ...)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(6, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
ecg_A = triage.ecg_array(A)          # 4,000 ECGs of 12 leads, 10 seconds at 100 Hz: 4000 x 12 x 1000
ecg_B = triage.ecg_array(B)

print(ecg_A.shape, ecg_B.shape)
i = int(np.where(A["patient_id"] == "CP-3304")[0][0])
t = np.arange(1000) / 100
fig, ax = plt.subplots(3, 1, figsize=(9, 4.2), sharex=True)
for a, lead in zip(ax, [1, 6, 10]):
    a.plot(t, ecg_A[i, lead], color="#231A1A", lw=0.8)
    a.set_ylabel(vclinic.LEADS[lead])
ax[-1].set_xlabel("seconds")
plt.show()

**Python note.** `triage.ecg_array` draws every ECG at 500 Hz and keeps every fifth sample, 100 per second, which is enough for the ST segment and the T wave and five times smaller to learn from. The arrays follow the rows of `A` and `B`, so `ecg_A[i]` is the ECG of the presentation in row `i` of `A`: `np.where(A["patient_id"] == "CP-3304")[0][0]` finds that row. Python counts from 0, so leads 1, 6 and 10 of the twelve are II, V1 and V5.

In [ ]:
class ECGNet(nn.Module):
    """A one-dimensional convolutional network for a 12-lead ECG: four blocks, the mean of every channel over time and one output."""
    def __init__(self):
        super().__init__()
        block = lambda a, b: nn.Sequential(nn.Conv1d(a, b, 7, padding=3), nn.BatchNorm1d(b), nn.ReLU(), nn.MaxPool1d(2))
        self.features = nn.Sequential(block(12, 32), block(32, 64), block(64, 64), block(64, 64))
        self.head = nn.Linear(64, 1)

    def embed(self, x):
        return self.features(x).mean(dim=2)

    def forward(self, x):
        return self.head(self.embed(x)).squeeze(1)


def train_ecg(model, X, y, Xv, yv, epochs=12, seed=2026, verbose=True):
    """Adam with a learning rate of 0.001 in groups of 64 ECGs, each shifted in time by up to half a second; the weights of the epoch with the best validation AUC are kept."""
    torch.manual_seed(seed)
    X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            b = order[i:i + 64]
            xb = torch.roll(X[b], int(torch.randint(-50, 51, (1,))), dims=2)
            loss = loss_fn(model(xb), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        a = roc_auc_score(yv, ecg_predict(model, Xv))
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
    model.load_state_dict(state)
    return best


@torch.no_grad()
def ecg_predict(model, X, batch=256):
    model.eval()
    return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


@torch.no_grad()
def ecg_embed(model, X, batch=256):
    model.eval()
    return torch.cat([model.embed(torch.as_tensor(X[i:i + batch])) for i in range(0, len(X), batch)]).numpy()

torch.manual_seed(RANDOM_STATE)
ecgnet = ECGNet()
print("weights:", sum(p.numel() for p in ecgnet.parameters()))
train_ecg(ecgnet, ecg_A[tr], y[tr], ecg_A[va], y[va])
P["ECG network"], PB["ECG network"] = ecg_predict(ecgnet, ecg_A), ecg_predict(ecgnet, ecg_B)
print("ECG network, test AUC", round(roc_auc_score(y[te], P["ECG network"][te]), 3), "| hospital B", round(roc_auc_score(yB, PB["ECG network"]), 3))
for reading in ["normal", "T-wave inversion", "ST depression", "ST elevation", "LBBB"]:
    m = (A["ecg_reading"] == reading).to_numpy() & te
    print(f"{reading:18} mean probability {P['ECG network'][m].mean():.3f}")

**Python note.** `nn.Conv1d(a, b, 7, padding=3)` slides 7 samples, 70 ms, along time over `a` channels and gives `b` channels; the first block reads the 12 leads. `nn.MaxPool1d(2)` halves the time after every block, so four blocks turn 1,000 samples into 62. `embed` averages every channel over time and gives 64 numbers, and `head` turns them into one logit. `train_ecg` shifts each tracing by up to half a second, `torch.roll`, so the network cannot learn where a beat falls; it keeps the weights of the epoch with the best validation AUC. The mean probability by reading shows what the network learned: The ST segment and the T wave move the risk.

**Observe and reflect.** The network never saw the reading of the ECG, only the tracing. Does its mean probability follow the readings in the order a cardiologist would expect? What would it need to read a tracing as well as a cardiologist?

**Exercise F.** How many trainable weights does the network for the ECG, `ecgnet`, hold?

In [ ]:
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in ecgnet.parameters())
teach.check("Exercise F", answer)

## 7. Early, late, hybrid and attention-based fusion

*Lecture: [Early fusion vs late fusion vs hybrid fusion](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#early-late-and-hybrid-fusion)*

This section builds five kinds of fusion on the same five modalities: early fusion, one logistic regression on all 103 numbers; late fusion by the mean of the five logits and by stacking; a hybrid network with one encoder per modality and a joint head; and an attention network with one token per modality, a class token, masks for a missing echocardiogram and modality dropout.

**From the formulas to the code.** $z = \beta_0 + \beta^{\top}[\,x_1; \ldots; x_5\,]$: `early`. $\frac{1}{5}\sum_m z_m$: `L.mean(axis=1)`. $b + \sum_m w_m z_m$: `stack`. $g([f_1(x_1); \ldots])$: `HybridFusion`. The mask $k_m$: `keep`; modality dropout: `drop=0.15`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(7, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
MODALITIES = ["clinical", "troponin", "echocardiogram", "ECG network", "note"]


class Attention(nn.Module):
    """Self-attention with several heads; a mask removes the tokens of missing modalities."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.weights = None

    def forward(self, x, keep):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        s = q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5
        s = s.masked_fill(~keep[:, None, None, :], float("-inf"))
        a = torch.softmax(s, dim=-1)
        self.weights = a.detach()
        return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    """A transformer block of Week 7, with the mask passed to its attention."""
    def __init__(self, dim, heads, hidden):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x, keep):
        x = x + self.attn(self.norm1(x), keep)
        return x + self.mlp(self.norm2(x))


class HybridFusion(nn.Module):
    """Intermediate fusion: one small encoder for each modality, their outputs side by side, and a joint head."""
    def __init__(self, dims, dim=32):
        super().__init__()
        self.enc = nn.ModuleList([nn.Sequential(nn.Linear(d, dim), nn.ReLU()) for d in dims])
        self.head = nn.Sequential(nn.Linear(dim * len(dims), dim), nn.ReLU(), nn.Dropout(0.2), nn.Linear(dim, 1))

    def forward(self, xs, keep):
        h = [e(x) * k[:, None] for e, x, k in zip(self.enc, xs, keep.T)]
        return self.head(torch.cat(h, dim=1)).squeeze(1)


class AttentionFusion(nn.Module):
    """Attention-based fusion: one token for each modality and a class token pass through two transformer blocks; missing modalities are masked."""
    def __init__(self, dims, dim=32, depth=2, heads=4):
        super().__init__()
        self.enc = nn.ModuleList([nn.Linear(d, dim) for d in dims])
        self.kind = nn.Parameter(torch.randn(1, len(dims), dim) * 0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
        self.norm, self.head = nn.LayerNorm(dim), nn.Linear(dim, 1)

    def forward(self, xs, keep):
        t = torch.stack([e(x) for e, x in zip(self.enc, xs)], dim=1) + self.kind
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1)
        keep = torch.cat([torch.ones(len(t), 1, dtype=torch.bool), keep], dim=1)
        for block in self.blocks:
            t = block(t, keep)
        return self.head(self.norm(t[:, 0])).squeeze(1)


def train_fusion(model, xs, keep, y, xs_v, keep_v, yv, epochs=40, seed=2026, drop=0.0, verbose=False):
    """Adam in groups of 64 patients; with drop > 0 each present modality is hidden at random during training (modality dropout). The weights of the epoch with the best validation AUC are kept."""
    torch.manual_seed(seed)
    xs = [torch.as_tensor(x, dtype=torch.float32) for x in xs]
    keep, y = torch.as_tensor(keep, dtype=torch.bool), torch.as_tensor(y, dtype=torch.float32)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y))
        for i in range(0, len(y), 64):
            b = order[i:i + 64]
            kb = keep[b]
            if drop > 0:
                kb = kb & (torch.rand(kb.shape) >= drop)
                kb[kb.sum(dim=1) == 0, 0] = True
            loss = loss_fn(model([x[b] for x in xs], kb), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        a = roc_auc_score(yv, fusion_predict(model, xs_v, keep_v))
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
    model.load_state_dict(state)
    return best


@torch.no_grad()
def fusion_predict(model, xs, keep):
    model.eval()
    xs = [torch.as_tensor(x, dtype=torch.float32) for x in xs]
    return torch.sigmoid(model(xs, torch.as_tensor(keep, dtype=torch.bool))).numpy()

emb = lambda X: pd.DataFrame(ecg_embed(ecgnet, X), columns=[f"ecg_{i + 1}" for i in range(64)])
parts_A = [clinical(A), laboratory(A), echo(A), emb(ecg_A), note(A)]
parts_B = [clinical(B), laboratory(B), echo(B), emb(ecg_B), note(B)]
XE, XEB = pd.concat(parts_A, axis=1), pd.concat(parts_B, axis=1)
early = fit_logistic(XE[tr], y[tr], Xv=XE[va], yv=y[va])
P["early"], PB["early"] = early.predict_proba(XE)[:, 1], early.predict_proba(XEB)[:, 1]
L, LB = np.column_stack([logit(P[m]) for m in MODALITIES]), np.column_stack([logit(PB[m]) for m in MODALITIES])
P["late"], PB["late"] = 1 / (1 + np.exp(-L.mean(axis=1))), 1 / (1 + np.exp(-LB.mean(axis=1)))
stack = LogisticRegression(max_iter=5000).fit(L[va], y[va])
P["stacked"], PB["stacked"] = stack.predict_proba(L)[:, 1], stack.predict_proba(LB)[:, 1]
print(XE.shape, "| stacking weights:", dict(zip(MODALITIES, np.round(stack.coef_[0], 3))), "| intercept", round(stack.intercept_[0], 3))

**Python note.** `pd.concat(parts_A, axis=1)` puts the five tables side by side into the row of early fusion: 12 + 4 + 3 + 64 + 20 = 103 numbers per presentation. The ECG enters early fusion as the 64 numbers of its network, and late fusion as the probability of the network. `L` holds the five logits of every presentation in the order of `MODALITIES`, and the same order gives the five weights of `stack`.

In [ ]:
scalers = [StandardScaler().fit(X[tr]) for X in parts_A]
xs = [s.transform(X).astype(np.float32) for s, X in zip(scalers, parts_A)]
xsB = [s.transform(X).astype(np.float32) for s, X in zip(scalers, parts_B)]
keep = np.ones((len(A), 5), dtype=bool)
keep[:, 2] = A["echo_done"].to_numpy() == 1
keepB = np.ones((len(B), 5), dtype=bool)
keepB[:, 2] = B["echo_done"].to_numpy() == 1
dims = [x.shape[1] for x in xs]
nets = {}
for name, Net, drop in (("hybrid", HybridFusion, 0.0), ("attention", AttentionFusion, 0.15)):
    torch.manual_seed(RANDOM_STATE)
    nets[name] = Net(dims)
    train_fusion(nets[name], [x[tr] for x in xs], keep[tr], y[tr], [x[va] for x in xs], keep[va], y[va], drop=drop)
    P[name], PB[name] = fusion_predict(nets[name], xs, keep), fusion_predict(nets[name], xsB, keepB)
    print(name, "| weights", sum(p.numel() for p in nets[name].parameters()), "| test AUC", round(roc_auc_score(y[te], P[name][te]), 3))

**Python note.** Each modality is standardised with its own training values, `StandardScaler().fit(X[tr])`. `keep` is the mask: a table of presentations × 5 modalities, `True` where the modality is present; only the echocardiogram can be missing here. The hybrid network multiplies the output of a missing encoder by 0, and the attention network gives the token of a missing modality the score minus infinity. `train_fusion` with `drop=0.15` also hides every present modality of a training batch with a probability of 15 percent, but never all of them. Both networks keep the weights of their best validation epoch.

In [ ]:
rows = []
for name in MODALITIES + ["early", "late", "stacked", "hybrid", "attention"]:
    lo, hi = np.percentile(bootstrap_auc(y[te], P[name][te]), [2.5, 97.5])
    rows.append([name, round(roc_auc_score(y[te], P[name][te]), 3), round(lo, 3), round(hi, 3)])
lo, hi = np.percentile(bootstrap_auc(y[te], heart[te].astype(float)), [2.5, 97.5])
rows.append(["HEART", round(auc_heart, 3), round(lo, 3), round(hi, 3)])
print(pd.DataFrame(rows, columns=["model", "test AUC", "2.5%", "97.5%"]).to_string(index=False))
i = int(np.where(A["patient_id"] == "CP-3304")[0][0])
print("CP-3304:", {name: round(float(P[name][i]), 3) for name in P})
d = paired_bootstrap(y[te], P["early"][te], heart[te].astype(float))
print("early fusion minus HEART", round(d.mean(), 3), np.round(np.percentile(d, [2.5, 97.5]), 3))

**Python note.** `bootstrap_auc` of Section 1 gives the 2,000 AUCs of each model and `np.percentile` their interval; `paired_bootstrap` compares early fusion with HEART in the same samples. The dictionary comprehension `{name: ... for name in P}` collects the probability of every model for the case, row `i` of `A`.

**Observe and reflect.** The five fusion models reach almost the same AUC. Which would you choose for a chest pain unit, and what besides the AUC would you look at before choosing? Compare the five probabilities of CP-3304.

**Exercise G.** How many trainable weights does the attention fusion network, `nets["attention"]`, hold?

In [ ]:
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in nets["attention"].parameters())
teach.check("Exercise G", answer)

## 8. Missing modalities at test time

*Lecture: [Missing data management: The clinical reality](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#missing-modalities)*

What happens when a whole modality is missing for every test patient, as when the laboratory system is down or the triage note was not written? Early fusion fills the modality with its training mean, the hybrid network multiplies the output of its encoder by 0, and the attention network masks its token, after training with modality dropout.

**From the formulas to the code.** The mask $k_m = 0$ for the hidden modality: `kk[:, j] = False`. The training mean in early fusion: `XE.loc[tr, cols].mean()`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(8, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
rows = []
for j, name in enumerate(MODALITIES):
    kk = keep.copy()
    kk[:, j] = False
    XE_ = XE.copy()
    cols = parts_A[j].columns
    XE_[cols] = XE.loc[tr, cols].mean().to_numpy()
    rows.append([name, round(roc_auc_score(y[te], early.predict_proba(XE_[te])[:, 1]), 3),
                 round(roc_auc_score(y[te], fusion_predict(nets["hybrid"], [x[te] for x in xs], kk[te])), 3),
                 round(roc_auc_score(y[te], fusion_predict(nets["attention"], [x[te] for x in xs], kk[te])), 3)])
missing = pd.DataFrame(rows, columns=["hidden", "early fusion", "hybrid", "attention"])
print(missing.to_string(index=False))
missing.set_index("hidden").plot.bar(figsize=(7, 3), ylim=(0.8, 0.95), rot=0, color=["#1F5F8B", "#0E7A78", "#6E46A0"])
plt.ylabel("test AUC")
plt.show()

**Python note.** `keep.copy()` makes a new mask so that the original stays untouched, and `kk[:, j] = False` hides modality `j` for every patient. `XE.copy()` does the same for the table of early fusion; assigning the training means to the columns of the hidden modality, `XE_[cols] = ...`, gives every patient the values of an average patient for it. The models themselves are not changed: This is a test of what each kind of fusion does with a gap that it meets for the first time at the bedside.

**Observe and reflect.** Which modality can the fusion models least afford to lose, and why? Would you trust a model that gives a probability when the troponin is missing, and what should the unit do instead?

**Exercise H.** What is the test AUC of early fusion when the triage note is hidden from every test patient? Read it from the table `missing` and keep it with three decimals.

In [ ]:
data = exercise_table("H")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = float(missing.loc[missing["hidden"] == "note", "early fusion"].iloc[0])
teach.check("Exercise H", answer)

**The Python path of Practice stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#another-hospital-drift-and-recalibration) and continue reading at «Another hospital: Drift and recalibration».

## 9. Calibration and another hospital

*Lecture: [Calibration: Do the probabilities mean what they say?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#calibration-do-the-probabilities-mean-what-they-say)*

The AUC judges the order of the patients, calibration the numbers. This section measures the calibration intercept, the slope and the Brier score of three fusion models on the test set of hospital A, draws their reliability curves, and then follows early fusion to hospital B, where it is updated with the first 300 patients.

**From the formulas to the code.** $\mathrm{logit}\,P(y = 1) = a + b\,z$: `calibration(y, p)` returns $a$ and $b$. Brier: `brier_score_loss(y, p)`. The update: the shift $a$ learned on `first`, added to every logit of B.

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(9, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.4))
for name, color in (("early", "#1F5F8B"), ("stacked", "#0E7A78"), ("late", "#B97813")):
    a, b = calibration(y[te], P[name][te])
    print(f"{name:8} mean {P[name][te].mean():.3f}  observed {y[te].mean():.3f}  intercept {a:+.2f}  slope {b:.2f}  Brier {brier_score_loss(y[te], P[name][te]):.3f}")
    pred, obs = reliability(y[te], P[name][te])
    ax.plot(pred, obs, "o-", color=color, label=name)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("mean predicted risk")
ax.set_ylabel("observed share of MACE")
ax.legend()
plt.show()

**Python note.** `calibration` finds the intercept $a$ by Newton steps with the slope fixed at 1: Each step adds the difference between the observed and the predicted number of events divided by the sum of $p(1 - p)$. The slope is the coefficient of a logistic regression of the outcome on the logit, without regularisation (`C=1e6`). `reliability` sorts the patients by their risk and splits them into ten groups of 80 with `np.array_split`, then returns the mean predicted risk and the observed share of every group.

In [ ]:
for name in ["early", "stacked", "hybrid", "attention"]:
    a, b = calibration(yB, PB[name])
    print(f"{name:10} hospital B: AUC {roc_auc_score(yB, PB[name]):.3f}  mean {PB[name].mean():.3f}  observed {yB.mean():.3f}  intercept {a:+.2f}  slope {b:.2f}")
first, rest = np.arange(len(B)) < 300, np.arange(len(B)) >= 300
zB = logit(PB["early"])
shift = 0.0
for _ in range(50):
    q = 1 / (1 + np.exp(-(shift + zB[first])))
    shift += (yB[first] - q).sum() / (q * (1 - q)).sum()
pB_new = 1 / (1 + np.exp(-(zB + shift)))
print("shift learned on the first 300:", round(shift, 3))
fig, ax = plt.subplots(figsize=(4.8, 4.4))
for p, label, color in ((PB["early"], "before", "#9B1B1B"), (pB_new, "after", "#1F5F8B")):
    a, b = calibration(yB[rest], p[rest])
    print(f"{label:7} mean {p[rest].mean():.3f}  observed {yB[rest].mean():.3f}  intercept {a:+.2f}  slope {b:.2f}  Brier {brier_score_loss(yB[rest], p[rest]):.3f}  AUC {roc_auc_score(yB[rest], p[rest]):.3f}")
    pred, obs = reliability(yB[rest], p[rest])
    ax.plot(pred, obs, "o-", color=color, label=label)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("mean predicted risk")
ax.set_ylabel("observed share of MACE")
ax.set_title("Hospital B, the 700 after the first 300", fontsize=10)
ax.legend()
plt.show()

**Python note.** The first 300 presentations of B, `first`, play the part of the first months of use; the other 700, `rest`, those after the update. The loop is the same Newton search as in `calibration`, but on the first 300 only: It finds the shift that makes their mean predicted risk equal to their share of events. Adding the shift to every logit of B, `zB + shift`, moves every patient up by the same step, so the order of the patients and the AUC do not change.

**Observe and reflect.** Every fusion model underestimates the risk at hospital B. Which of the four differences of B could the inputs show, and which could they not? Why does the update close only part of the gap?

**Exercise I.** What is the Brier score of early fusion on the 700 later patients of hospital B after the update, `pB_new[rest]`? Keep it with three decimals.

In [ ]:
data = exercise_table("I")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(brier_score_loss(yB[rest], pB_new[rest])), 3)
teach.check("Exercise I", answer)

## 10. Thresholds for discharge and net benefit

*Lecture: [Thresholds for discharge and net benefit](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#thresholds-for-discharge-and-net-benefit)*

Two thresholds of early fusion are chosen on the validation presentations: one that sends home as many patients as HEART 0 to 3, and one that keeps the sensitivity at 98 percent. Both are tested next to HEART 0 to 3 on the test presentations and at hospital B, with Wilson intervals for the share of MACE among the patients sent home. A decision curve then compares the model with HEART for the decision to admit.

**From the formulas to the code.** The thresholds: `share_threshold(p, k)` and `rule_out_threshold(y, p, 0.98)`. The Wilson interval: `wilson(k, n)`. $\mathrm{NB}(t) = \mathrm{TP}/n - \mathrm{FP}/n \cdot t/(1 - t)$: `net_benefit(y, p, thresholds)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB08_multimodal_ai_and_data_fusion.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(10, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
def share_threshold(p, k):
    """The threshold that sends home the k patients with the lowest predicted risk: the k-th lowest risk."""
    return float(np.sort(p)[k - 1])


def rule_out_threshold(y, p, sensitivity):
    """The largest threshold that keeps the sensitivity for the events at least at the given level. Patients at or below it are discharged. It is chosen on the validation set."""
    order = np.argsort(p)
    missed = np.cumsum(y[order])              # events at or below each threshold
    ok = np.where(missed <= (1 - sensitivity) * y.sum())[0]
    return float(p[order][ok.max()]) if len(ok) else 0.0


def wilson(k, n, z=1.96):
    """The 95 percent Wilson interval of a proportion k/n."""
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (c - h, c + h)


def net_benefit(y, p, thresholds):
    """Net benefit of treating the patients over each threshold: true positives minus false positives weighted by the odds of the threshold, per patient."""
    n = len(y)
    return np.array([((p >= t) & (y == 1)).sum() / n - ((p >= t) & (y == 0)).sum() / n * t / (1 - t) for t in thresholds])


def discharge(home, yy, label):
    k, n = int(yy[home].sum()), int(home.sum())
    lo, hi = wilson(k, n)
    print(f"{label:34} home: {n:4} ({home.mean():.3f})  MACE: {k:3}  ({k / n:.3f}; {lo:.3f}-{hi:.3f})  sensitivity {1 - k / yy.sum():.3f}")


k_heart = int((heart[va] <= 3).sum())
t_share = share_threshold(P["early"][va], k_heart)
t_98 = rule_out_threshold(y[va], P["early"][va], 0.98)
print("thresholds:", round(t_share, 4), round(t_98, 4))
discharge(heart[te] <= 3, y[te], "HEART 0-3")
discharge(P["early"][te] <= t_share, y[te], "early fusion, same share")
discharge(P["early"][te] <= t_98, y[te], "early fusion, 98 percent")
print("Hospital B, the 700 after the first 300:")
discharge(B["heart"].to_numpy()[rest] <= 3, yB[rest], "HEART 0-3")
discharge(PB["early"][rest] <= t_share, yB[rest], "threshold of A, before the update")
discharge(pB_new[rest] <= t_share, yB[rest], "threshold of A, after the update")

**Python note.** `share_threshold` sorts the validation risks and takes the k-th lowest, where k is the number of validation patients with HEART 0 to 3. `rule_out_threshold` walks up the sorted risks and counts the events that fall at or below each one, `np.cumsum`, and keeps the largest threshold at which no more than 2 percent of the events would be sent home. `discharge` is a small function of this cell that prints one line for a rule: how many go home, their share, the MACE among them with the Wilson interval, and the sensitivity, the share of all events kept in hospital.

In [ ]:
ths = np.round(np.arange(0.01, 0.51, 0.01), 2)
curves = {"early fusion": net_benefit(y[te], P["early"][te], ths),
          "HEART ≥ 4": net_benefit(y[te], (heart[te] >= 4).astype(float), ths),
          "admit everyone": y[te].mean() - (1 - y[te].mean()) * ths / (1 - ths)}
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for (name, nb), color in zip(curves.items(), ["#1F5F8B", "#B97813", "#8A8A8A"]):
    ax.plot(ths, nb, color=color, label=name)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylim(-0.05, 0.2)
ax.set_xlabel("threshold probability")
ax.set_ylabel("net benefit")
ax.legend()
plt.show()
for t in (0.05, 0.10, 0.20, 0.30):
    j = int(round(t * 100)) - 1
    print(f"t = {t:.2f}: " + "  ".join(f"{name} {nb[j]:.3f}" for name, nb in curves.items()))

**Python note.** `np.arange(0.01, 0.51, 0.01)` gives the thresholds from 1 to 50 percent. For HEART, a rule of yes or no, the array `(heart[te] >= 4).astype(float)` serves as the probability: 1 for every patient admitted, 0 for the others, so it is admitted at every threshold. The net benefit of admitting everyone needs no model: the share of events minus the share without events times the odds of the threshold. `nb[j]` reads the curve at threshold `t`, since the thresholds start at 0.01 and rise by 0.01.

**Observe and reflect.** For the decision to discharge, the model is about as good as HEART; for the decision to admit at 20 or 30 percent it is clearly better. How would you explain this to a heart team, and which decision would you trust the model with first?

**Exercise J.** What is the net benefit of early fusion on the test presentations at a threshold probability of 10 percent? Keep it with three decimals.

In [ ]:
data = exercise_table("J")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(net_benefit(y[te], P["early"][te], [0.10])[0]), 3)
teach.check("Exercise J", answer)

**The Python path of Practice stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lecture.html#ai-pair-programming-part-8-reviewing-a-fusion-pipeline) and continue reading at «Working with an AI assistant, part 8: Reviewing a fusion pipeline». The rest of this Jupyter notebook holds further practice and the help for the weekly project.

## Application challenges by field

| Field | Challenge |
|---|---|
| Emergency cardiology | Build the HEART pathway of the guidelines in the chest pain unit: HEART 0 to 3 and both troponins at or below the upper reference limit. How many test patients would it send home, and how many MACE would it miss, next to HEART 0 to 3 alone? |
| Interventional cardiology | Look at the MACE of hospital B by type. How much of the extra risk at B comes from revascularisations in angina, and what would a model that predicted only myocardial infarction and death show at B? |
| Heart failure | In the registry, add the NT-proBNP of the laboratory to the echocardiogram in one model and leave the ECG out. Is this model of two modalities better than early fusion of all three, and how sure is the difference? |
| Clinical research | Repeat the update of hospital B with the first 100, 200, 300, 500 and 700 patients, each time testing on the presentations after them. How does the calibration intercept on the later patients change with the number of local patients? |
| Women and the heart | Compare HEART and early fusion in women and in men among the test presentations: the AUC, the calibration intercept and the MACE missed at the threshold of the same share. Does either rule miss more events in women? |
| Quality and safety | List the ten test patients sent home by early fusion who had a MACE. What were their diagnosis, their troponins, their HEART score and their note? Which of them could the team have caught, and how? |
| Medical informatics | Write the specification of a data extract for a real chest pain unit: the systems, the index time, every input with its unit and time stamp, the definition of MACE and the sources of follow-up, and the checks before the first model. |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-08/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of this week extends Sections 7, 9 and 10: The models are compared with HEART with intervals, their calibration is judged at both hospitals, two thresholds for discharge are tested, and the decision curve is drawn for a heart team.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>